# P7513 MERSCOPE: ProSeg hybrid subsection test

This notebook builds a small, viewer-ready SpatialData store from an editable P7513 MERSCOPE XY box and runs MerXen's production `proseg_hybrid` refinement on it. It preserves ProSeg foreground/background calls and quality columns, uses all ProSeg-assigned transcripts for the selected cells (including remote candidates), constructs the transcript-supported concave segmentations, and then reruns assignment with the single-mask/ProSeg-overlap rule.

Edit only the **Settings** cell for the usual test. The output store contains the Cellpose source masks, original ProSeg masks, hybrid masks, original point annotations, hybrid assignment/provenance columns, and the hybrid count table. Cells whose Cellpose masks are truncated by the context crop are excluded.

In [ ]:
from __future__ import annotations

import json
import shutil
import sys
from collections import deque
from itertools import product
from pathlib import Path

import anndata as ad
import dask.dataframe as dd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import spatialdata as sd
from IPython.display import display
from scipy.spatial import cKDTree
from shapely.geometry import MultiPoint, Point, Polygon, box
from shapely.ops import unary_union
from spatialdata.models import Image2DModel, PointsModel, ShapesModel, TableModel
from spatialdata.transformations import Affine, set_transformation

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "merxen").exists():
    REPO_ROOT = next(
        parent for parent in Path.cwd().resolve().parents
        if (parent / "src" / "merxen").exists()
    )
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from merxen.config import ProsegHybridConfig
from merxen.io.image_source import (
    MERSCOPE_ZPROJ_IMAGE_NAME,
    build_merscope_z_projection,
)
from merxen.io.spatialdata_io import write_spatialdata_zarr
from merxen.segmentation.cellpose import invert_mask_affine
from merxen.segmentation.proseg_hybrid import (
    HYBRID_ASSIGNMENT_COLUMN,
    HYBRID_ASSIGNMENT_SOURCE_COLUMN,
    PROSEG_HYBRID_SHAPE_NAME,
    PROSEG_HYBRID_TABLE_NAME,
    _cellpose_polygons_in_microns,
    assign_transcripts_to_hybrid_masks,
    build_hybrid_cell_geometry,
    run_proseg_hybrid_refinement,
    select_bulk_transcripts,
)
from merxen.visualization.sanity_plots import _resolve_dataset_mask_affine


In [ ]:
# ----------------------------- Settings ---------------------------------
SOURCE_ZARR = (
    REPO_ROOT / "results" / "P7513" / "merscope"
    / "latest" / "latest_spatialdata.zarr"
)
CELLPOSE_MASK_PATH = (
    REPO_ROOT / "results" / "P7513" / "merscope"
    / "segmentation" / "cellpose_masks_tiled.npy"
)

# (x_min, y_min, x_max, y_max), in MERSCOPE microns.
REGION_BBOX_UM = (5500.0, 5230.0, 5650.0, 5380.0)

# Extra mask/transcript context makes cells selected at the box edge complete.
MASK_CONTEXT_UM = 25.0

# True is important when testing robust rejection of spuriously remote assignments.
KEEP_ALL_PROSEG_ASSIGNED_TRANSCRIPTS_FOR_SELECTED_CELLS = True

# P7513 predates the loader change, so its old store may call the score `qv`.
# Either `transcript_score` or `qv` is retained. Control probes are excluded.
DROP_MERSCOPE_CONTROLS = True
CONTROL_GENE_PATTERN = (
    r"^(Blank($|-)|UnassignedCodeword|NegControlCodeword|NegControlProbe)"
)

HYBRID_CONFIG = ProsegHybridConfig(
    enabled=True,
    min_transcripts=10,
    outlier_neighbors=3,
    outlier_mad_multiplier=4.0,
    concavity_ratio=0.2,
    smoothing_scale=0.5,
)

# Outlier sweep. Lower MAD and lower k are harsher. Hull parameters stay
# fixed so the effect of outlier selection can be compared directly.
SWEEP_OUTLIER_NEIGHBORS = (3, 2)
SWEEP_OUTLIER_MAD_MULTIPLIERS = (4.0, 3.0, 2.5, 2.0)
SWEEP_CONCAVITY_RATIO = HYBRID_CONFIG.concavity_ratio
SWEEP_SMOOTHING_SCALE = HYBRID_CONFIG.smoothing_scale
SWEEP_DISPLAY_BBOX_UM = REGION_BBOX_UM  # Replace with a tighter XY box to zoom in.
SWEEP_N_COLUMNS = 4

# Supported-envelope experiment. This holds the successful harsh outlier
# setting fixed, then sweeps the external-support graph parameters.
SUPPORTED_BASE_OUTLIER_NEIGHBORS = 2
SUPPORTED_BASE_OUTLIER_MAD_MULTIPLIER = 2.0
SUPPORTED_MIN_NEIGHBORS = (2, 3, 4)
SUPPORTED_RADIUS_SCALES = (1.5, 2.0)
SUPPORTED_ENVELOPE_RADIUS_SCALE = 1.0
SUPPORTED_OPENING_SCALE = 0.5
SUPPORTED_CLOSING_SCALE = 0.75
SUPPORTED_DISPLAY_BBOX_UM = SWEEP_DISPLAY_BBOX_UM
SUPPORTED_N_COLUMNS = 3

# Distance-weighted envelope experiment. A nearby assigned singleton can
# receive Cellpose-surface support, while absolute distance progressively
# suppresses even locally dense transcript chains.
DISTANCE_BASE_OUTLIER_NEIGHBORS = 2
DISTANCE_BASE_OUTLIER_MAD_MULTIPLIER = 2.0
DISTANCE_DECAY_RADIUS_FRACTIONS = (0.25, 0.5, 0.75)
DISTANCE_SURFACE_REWARDS = (2.0, 3.0)
DISTANCE_LOCAL_SUPPORT_RADIUS_SCALE = 2.0
DISTANCE_MIN_EFFECTIVE_SUPPORT = 2.0
DISTANCE_ENVELOPE_RADIUS_SCALE = 1.0
DISTANCE_OPENING_SCALE = 0.5
DISTANCE_CLOSING_SCALE = 0.75
DISTANCE_DISPLAY_BBOX_UM = SUPPORTED_DISPLAY_BBOX_UM
DISTANCE_N_COLUMNS = 3

OUTPUT_DIR = REPO_ROOT / "results" / "P7513" / "notebook_outputs"
OUTPUT_ZARR = OUTPUT_DIR / "P7513_proseg_hybrid_subsection.spatialdata.zarr"
CROPPED_MASK_PATH = OUTPUT_DIR / "P7513_hybrid_subsection_cellpose.npy"
CROPPED_TRANSFORM_PATH = OUTPUT_DIR / "P7513_hybrid_subsection_transform.json"
COMPARISON_PLOT_PATH = OUTPUT_DIR / "P7513_proseg_hybrid_subsection_comparison.png"
DIAGNOSTIC_PLOT_PATH = OUTPUT_DIR / "P7513_proseg_hybrid_subsection_diagnostics.png"
SWEEP_PLOT_PATH = OUTPUT_DIR / "P7513_proseg_hybrid_outlier_sweep.png"
SWEEP_SUMMARY_PATH = OUTPUT_DIR / "P7513_proseg_hybrid_outlier_sweep.csv"
SUPPORTED_SWEEP_PLOT_PATH = OUTPUT_DIR / "P7513_supported_envelope_sweep.png"
SUPPORTED_SWEEP_SUMMARY_PATH = OUTPUT_DIR / "P7513_supported_envelope_sweep.csv"
DISTANCE_SWEEP_PLOT_PATH = OUTPUT_DIR / "P7513_distance_weighted_envelope_sweep.png"
DISTANCE_SWEEP_SUMMARY_PATH = OUTPUT_DIR / "P7513_distance_weighted_envelope_sweep.csv"
CELL_SUMMARY_PATH = OUTPUT_DIR / "P7513_proseg_hybrid_subsection_cells.csv"
# ------------------------------------------------------------------------

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Source: {SOURCE_ZARR}")
print(f"Requested region: {REGION_BBOX_UM}")
print(f"Output: {OUTPUT_ZARR}")


## Subsection construction

The requested box selects Cellpose labels. The raw mask is cropped with context, but labels not selected by the requested box are zeroed, and labels touching the context edge are omitted. For those cells, the point subset contains every transcript in the context plus every transcript assigned to the selected cells by ProSeg. This deliberately retains distant bad assignments until the robust bulk-selection step has had a chance to reject them.

In [ ]:
def bbox_to_pixel_window(
    bbox_um: tuple[float, float, float, float],
    x_transform: tuple[float, float, float],
    y_transform: tuple[float, float, float],
    image_shape: tuple[int, int],
    *,
    padding_um: float = 0.0,
) -> tuple[int, int, int, int]:
    """Return clipped (row_start, row_stop, col_start, col_stop)."""
    x_min, y_min, x_max, y_max = bbox_um
    corners_um = np.asarray(
        [
            [x_min - padding_um, y_min - padding_um],
            [x_min - padding_um, y_max + padding_um],
            [x_max + padding_um, y_min - padding_um],
            [x_max + padding_um, y_max + padding_um],
        ],
        dtype=np.float64,
    )
    inverse, offset = invert_mask_affine(x_transform, y_transform)
    corners_px = (inverse @ (corners_um - offset).T).T
    col_start = max(0, int(np.floor(corners_px[:, 0].min())))
    col_stop = min(image_shape[1], int(np.ceil(corners_px[:, 0].max())) + 1)
    row_start = max(0, int(np.floor(corners_px[:, 1].min())))
    row_stop = min(image_shape[0], int(np.ceil(corners_px[:, 1].max())) + 1)
    return row_start, row_stop, col_start, col_stop


def shifted_crop_affine(
    x_transform: tuple[float, float, float],
    y_transform: tuple[float, float, float],
    row_start: int,
    col_start: int,
) -> tuple[tuple[float, float, float], tuple[float, float, float]]:
    """Shift a full-mask affine to the origin of a cropped mask."""
    x_shifted = (
        x_transform[0],
        x_transform[1],
        x_transform[0] * col_start + x_transform[1] * row_start + x_transform[2],
    )
    y_shifted = (
        y_transform[0],
        y_transform[1],
        y_transform[0] * col_start + y_transform[1] * row_start + y_transform[2],
    )
    return x_shifted, y_shifted


def edge_labels(mask: np.ndarray) -> np.ndarray:
    """Return nonzero labels touching a crop edge."""
    values = np.concatenate([mask[0], mask[-1], mask[:, 0], mask[:, -1]])
    return np.unique(values[values > 0])


def plot_polygons_by_cell(
    ax: plt.Axes,
    frame: gpd.GeoDataFrame,
    ids: pd.Series,
    colors: dict[str, tuple[float, float, float, float]],
    *,
    alpha: float = 0.25,
) -> None:
    """Plot polygons with one stable color per canonical Cellpose label."""
    facecolors = [colors.get(str(value), (0.6, 0.6, 0.6, 1.0)) for value in ids]
    edgecolors = [colors.get(str(value), (0.3, 0.3, 0.3, 1.0)) for value in ids]
    frame.plot(ax=ax, color=facecolors, edgecolor=edgecolors, alpha=alpha, linewidth=0.8)


def plot_points_by_cell(
    ax: plt.Axes,
    frame: pd.DataFrame,
    cell_column: str,
    colors: dict[str, tuple[float, float, float, float]],
    *,
    size: float = 3.0,
    alpha: float = 0.85,
    marker: str = "o",
) -> None:
    """Plot assigned transcripts using the matching cell color."""
    assigned = frame[cell_column].notna()
    if not bool(assigned.any()):
        return
    selected = frame.loc[assigned]
    point_colors = [colors[str(value)] for value in selected[cell_column].astype(str)]
    x_column = "observed_x" if "observed_x" in selected else "x"
    y_column = "observed_y" if "observed_y" in selected else "y"
    linewidths = 0.8 if marker in {"+", "x"} else 0.0
    ax.scatter(
        selected[x_column], selected[y_column], c=point_colors,
        s=size, alpha=alpha, marker=marker, linewidths=linewidths,
    )


def plot_sweep_transcripts(
    ax: plt.Axes,
    frame: pd.DataFrame,
    colors: dict[str, tuple[float, float, float, float]],
) -> None:
    """Plot final calls while preserving original ProSeg assignment status."""
    x_column = "observed_x" if "observed_x" in frame else "x"
    y_column = "observed_y" if "observed_y" in frame else "y"
    originally_unassigned = frame["background"].astype(bool)
    finally_assigned = frame[HYBRID_ASSIGNMENT_COLUMN].notna()

    plot_points_by_cell(
        ax,
        frame.loc[finally_assigned & ~originally_unassigned],
        HYBRID_ASSIGNMENT_COLUMN,
        colors,
        size=3.0,
        alpha=0.88,
        marker="o",
    )
    plot_points_by_cell(
        ax,
        frame.loc[finally_assigned & originally_unassigned],
        HYBRID_ASSIGNMENT_COLUMN,
        colors,
        size=10.0,
        alpha=0.9,
        marker="+",
    )

    unresolved_background = originally_unassigned & ~finally_assigned
    ax.scatter(
        frame.loc[unresolved_background, x_column],
        frame.loc[unresolved_background, y_column],
        c="0.68", marker="+", s=10, alpha=0.8, linewidths=0.65, zorder=2,
    )
    rejected_foreground = (
        ~originally_unassigned
        & frame["proseg_cell_id"].notna()
        & ~finally_assigned
    )
    ax.scatter(
        frame.loc[rejected_foreground, x_column],
        frame.loc[rejected_foreground, y_column],
        c="black", marker="x", s=14, linewidths=0.65, zorder=5,
    )
    other_unassigned = ~originally_unassigned & ~finally_assigned & ~rejected_foreground
    ax.scatter(
        frame.loc[other_unassigned, x_column],
        frame.loc[other_unassigned, y_column],
        c="0.78", marker="o", s=1.2, alpha=0.25, linewidths=0, zorder=2,
    )


In [ ]:
assert SOURCE_ZARR.exists(), SOURCE_ZARR
assert CELLPOSE_MASK_PATH.exists(), CELLPOSE_MASK_PATH
x_min, y_min, x_max, y_max = REGION_BBOX_UM
assert x_min < x_max and y_min < y_max

x_transform, y_transform = _resolve_dataset_mask_affine(
    "MERSCOPE", zarr_path=SOURCE_ZARR
)
full_mask = np.load(CELLPOSE_MASK_PATH, mmap_mode="r")
context_window = bbox_to_pixel_window(
    REGION_BBOX_UM, x_transform, y_transform, full_mask.shape,
    padding_um=MASK_CONTEXT_UM,
)
core_window = bbox_to_pixel_window(
    REGION_BBOX_UM, x_transform, y_transform, full_mask.shape
)
row_start, row_stop, col_start, col_stop = context_window
core_row_start, core_row_stop, core_col_start, core_col_stop = core_window

candidate_labels = np.unique(
    full_mask[core_row_start:core_row_stop, core_col_start:core_col_stop]
)
candidate_labels = candidate_labels[candidate_labels > 0]
mask_crop = np.asarray(
    full_mask[row_start:row_stop, col_start:col_stop]
).copy()
truncated_labels = np.intersect1d(candidate_labels, edge_labels(mask_crop))
interior_labels = np.setdiff1d(candidate_labels, truncated_labels)

source_proseg_table = ad.read_zarr(
    SOURCE_ZARR / "tables" / "table_MOSAIK_proseg"
)
known_original_ids = set(source_proseg_table.obs["original_cell_id"].astype(str))
selected_labels = np.asarray(
    [label for label in interior_labels if str(int(label)) in known_original_ids],
    dtype=np.uint32,
)
if len(selected_labels) == 0:
    raise RuntimeError("The requested box contains no complete Cellpose/ProSeg cells.")

mask_crop[~np.isin(mask_crop, selected_labels)] = 0
np.save(CROPPED_MASK_PATH, mask_crop)
crop_x_transform, crop_y_transform = shifted_crop_affine(
    x_transform, y_transform, row_start, col_start
)
CROPPED_TRANSFORM_PATH.write_text(
    json.dumps(
        {
            "x_transform": list(crop_x_transform),
            "y_transform": list(crop_y_transform),
            "source_pixel_window": list(context_window),
            "requested_bbox_um": list(REGION_BBOX_UM),
        },
        indent=2,
    )
)

selected_original_ids = [str(int(label)) for label in selected_labels]
proseg_row_mask = source_proseg_table.obs["original_cell_id"].astype(str).isin(
    selected_original_ids
)
selected_proseg_table = source_proseg_table[proseg_row_mask].copy()
selected_proseg_ids = selected_proseg_table.obs["cell"].astype(np.int64).tolist()

print(f"Mask affine: x={x_transform}, y={y_transform}")
print(f"Context mask crop: {mask_crop.shape} pixels; window={context_window}")
print(f"Candidate labels in requested box: {len(candidate_labels):,}")
print(f"Excluded because context clipped the mask: {len(truncated_labels):,}")
print(f"Selected Cellpose/ProSeg cells: {len(selected_labels):,}")


In [ ]:
# Load only the lazy transcript layer; do not materialize the full enriched store.
source_points_sdata = sd.read_zarr(SOURCE_ZARR, selection=("points",))
source_points = source_points_sdata.points["transcripts"]
required_columns = {"observed_x", "observed_y", "gene", "assignment", "background"}
missing_columns = required_columns.difference(source_points.columns)
if missing_columns:
    raise KeyError(f"Source transcript layer is missing {sorted(missing_columns)}")

in_context = (
    source_points["observed_x"].between(x_min - MASK_CONTEXT_UM, x_max + MASK_CONTEXT_UM)
    & source_points["observed_y"].between(y_min - MASK_CONTEXT_UM, y_max + MASK_CONTEXT_UM)
)
assigned_to_selected = source_points["assignment"].isin(selected_proseg_ids).fillna(False)
point_selector = in_context
if KEEP_ALL_PROSEG_ASSIGNED_TRANSCRIPTS_FOR_SELECTED_CELLS:
    point_selector = point_selector | assigned_to_selected
if DROP_MERSCOPE_CONTROLS:
    controls = source_points["gene"].astype("string").str.match(
        CONTROL_GENE_PATTERN, na=False
    )
    point_selector = point_selector & ~controls

points_pdf = source_points.loc[point_selector].compute().reset_index(drop=True)
if len(points_pdf) == 0:
    raise RuntimeError("No transcripts were found for the selected subsection.")

score_columns = [column for column in ["transcript_score", "qv"] if column in points_pdf]
print(f"Subsection transcripts retained: {len(points_pdf):,}")
print(f"Quality/score columns retained: {score_columns or 'none in this old store'}")
print(f"ProSeg foreground transcripts: {(~points_pdf['background'].astype(bool)).sum():,}")

# Build the source Cellpose geometry using the same conversion called by production.
cellpose_polygons = _cellpose_polygons_in_microns(
    CROPPED_MASK_PATH, crop_x_transform, crop_y_transform
)
cellpose_ids = sorted(cellpose_polygons, key=int)
cellpose_gdf = gpd.GeoDataFrame(
    {
        "cell_id": cellpose_ids,
        "cellpose_label": [int(value) for value in cellpose_ids],
        "geometry": [cellpose_polygons[value] for value in cellpose_ids],
    },
    geometry="geometry",
    index=pd.Index(cellpose_ids, name="cell_id_index"),
)

# Predicate pushdown reads just the selected original ProSeg polygons.
proseg_gdf = gpd.read_parquet(
    SOURCE_ZARR / "shapes" / "MOSAIK_proseg" / "shapes.parquet",
    filters=[("cell", "in", selected_proseg_ids)],
)
proseg_gdf = proseg_gdf.set_index(
    pd.Index(proseg_gdf["cell"].astype(np.int64), name="cell"),
    drop=False,
)

# The production routine expects its ProSeg mapping/count table under `table`.
if DROP_MERSCOPE_CONTROLS:
    gene_names = selected_proseg_table.var_names.astype(str)
    gene_keep = ~pd.Series(gene_names).str.match(CONTROL_GENE_PATTERN, na=False).to_numpy()
    selected_proseg_table = selected_proseg_table[:, gene_keep].copy()
selected_proseg_table.obs["region"] = pd.Categorical(
    ["MOSAIK_proseg_source"] * selected_proseg_table.n_obs,
    categories=["MOSAIK_proseg_source"],
)
source_table = TableModel.parse(
    selected_proseg_table,
    region="MOSAIK_proseg_source",
    region_key="region",
    instance_key="cell",
    overwrite_metadata=True,
)

npartitions = max(1, min(4, int(np.ceil(len(points_pdf) / 50_000))))
parsed_points = PointsModel.parse(
    dd.from_pandas(points_pdf, npartitions=npartitions),
    coordinates={"x": "observed_x", "y": "observed_y", "z": "observed_z"},
    feature_key="gene",
)
source_test_sdata = sd.SpatialData(
    points={"transcripts": parsed_points},
    shapes={
        "MOSAIK_cellpose_source": ShapesModel.parse(cellpose_gdf),
        "MOSAIK_proseg_source": ShapesModel.parse(proseg_gdf),
    },
    tables={"table": source_table},
)

if OUTPUT_ZARR.exists():
    shutil.rmtree(OUTPUT_ZARR)
write_spatialdata_zarr(source_test_sdata, OUTPUT_ZARR)
print(f"Wrote source subsection: {OUTPUT_ZARR}")


## Run the production hybrid branch

This is the same entry point used by the SEGMENT workflow. It collects ProSeg-foreground coordinates per cell, performs robust component/outlier selection, builds and smooths the concave transcript hull, unions it with Cellpose, writes GeoParquet-backed SpatialData polygons, and reassigns every retained point.

In [ ]:
run_summary = run_proseg_hybrid_refinement(
    OUTPUT_ZARR,
    CROPPED_MASK_PATH,
    CROPPED_TRANSFORM_PATH,
    HYBRID_CONFIG,
)
display(pd.Series(run_summary, name="value").to_frame())


In [ ]:
result = sd.read_zarr(OUTPUT_ZARR)
result_points = result.points["transcripts"].compute()
hybrid_gdf = result.shapes[PROSEG_HYBRID_SHAPE_NAME].copy()
hybrid_table = result.tables[PROSEG_HYBRID_TABLE_NAME]

assignment_map = {
    int(cell): str(original)
    for cell, original in zip(
        selected_proseg_table.obs["cell"],
        selected_proseg_table.obs["original_cell_id"],
        strict=True,
    )
}
result_points["proseg_cell_id"] = (
    pd.to_numeric(result_points["assignment"], errors="coerce")
    .map(assignment_map)
    .astype("string")
)
proseg_gdf["cell_id"] = proseg_gdf["cell"].map(assignment_map).astype("string")

result_x_column = "observed_x" if "observed_x" in result_points else "x"
result_y_column = "observed_y" if "observed_y" in result_points else "y"
display_mask = (
    result_points[result_x_column].between(x_min, x_max)
    & result_points[result_y_column].between(y_min, y_max)
)
display_points = result_points.loc[display_mask].copy()
cell_ids = sorted(hybrid_gdf.index.astype(str), key=int)
cmap = plt.colormaps["turbo"].resampled(max(1, len(cell_ids)))
palette_order = np.random.default_rng(7513).permutation(len(cell_ids))
cell_colors = {
    cell_id: cmap(int(palette_index))
    for cell_id, palette_index in zip(cell_ids, palette_order, strict=True)
}

hybrid_obs = hybrid_table.obs.copy()
hybrid_obs.to_csv(CELL_SUMMARY_PATH)
print(f"Points displayed inside requested box: {len(display_points):,}")
print(f"Hybrid cells: {len(hybrid_gdf):,}")
print(f"Per-cell diagnostics: {CELL_SUMMARY_PATH}")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 16), constrained_layout=True)
axes = axes.ravel()

foreground = ~display_points["background"].astype(bool)
newly_assigned = (
    display_points["background"].astype(bool)
    & display_points[HYBRID_ASSIGNMENT_COLUMN].notna()
)
rejected_proseg = (
    foreground
    & display_points["proseg_cell_id"].notna()
    & display_points[HYBRID_ASSIGNMENT_COLUMN].isna()
)

plot_polygons_by_cell(
    axes[0], cellpose_gdf, cellpose_gdf["cell_id"], cell_colors
)
plot_points_by_cell(
    axes[0], display_points.loc[foreground], "proseg_cell_id", cell_colors
)
axes[0].set_title("Cellpose masks + ProSeg-foreground transcripts")

plot_polygons_by_cell(
    axes[1], proseg_gdf, proseg_gdf["cell_id"], cell_colors
)
plot_points_by_cell(
    axes[1], display_points.loc[foreground], "proseg_cell_id", cell_colors
)
axes[1].set_title("Original ProSeg masks + ProSeg-foreground transcripts")

plot_polygons_by_cell(
    axes[2], hybrid_gdf, pd.Series(hybrid_gdf.index, index=hybrid_gdf.index), cell_colors
)
unassigned = display_points[HYBRID_ASSIGNMENT_COLUMN].isna()
axes[2].scatter(
    display_points.loc[unassigned, result_x_column],
    display_points.loc[unassigned, result_y_column],
    c="0.75", s=1.5, alpha=0.35, linewidths=0,
)
plot_points_by_cell(
    axes[2], display_points, HYBRID_ASSIGNMENT_COLUMN, cell_colors
)
axes[2].set_title("Hybrid masks + final transcript assignments")

plot_polygons_by_cell(
    axes[3], hybrid_gdf, pd.Series(hybrid_gdf.index, index=hybrid_gdf.index), cell_colors, alpha=0.12
)
plot_points_by_cell(
    axes[3], display_points.loc[newly_assigned], HYBRID_ASSIGNMENT_COLUMN, cell_colors, size=7.0
)
axes[3].scatter(
    display_points.loc[rejected_proseg, result_x_column],
    display_points.loc[rejected_proseg, result_y_column],
    c="black", marker="x", s=18, linewidths=0.7,
)
axes[3].set_title(
    "Changed calls: background rescued (cell colours), ProSeg rejected (black ×)"
)

for ax in axes:
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_aspect("equal")
    ax.set_xlabel("observed x (µm)")
    ax.set_ylabel("observed y (µm)")
    ax.grid(False)

fig.suptitle("P7513 MERSCOPE: Cellpose → ProSeg → hybrid refinement", fontsize=16)
fig.savefig(COMPARISON_PLOT_PATH, dpi=220, bbox_inches="tight")
plt.show()
print(f"Saved {COMPARISON_PLOT_PATH}")


In [ ]:
diagnostics = hybrid_obs.copy()
diagnostics["area_ratio"] = (
    diagnostics["hybrid_area_um2"] / diagnostics["cellpose_area_um2"]
)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), constrained_layout=True)
axes[0].hist(diagnostics["area_ratio"], bins=30, color="#4C78A8")
axes[0].axvline(1.0, color="black", linestyle="--", linewidth=1)
axes[0].set(title="Hybrid / Cellpose area", xlabel="area ratio", ylabel="cells")

axes[1].scatter(
    diagnostics["proseg_foreground_transcripts"],
    diagnostics["outlier_transcripts"],
    s=10, alpha=0.65, color="#F58518",
)
axes[1].set(
    title="Robust outlier filtering",
    xlabel="ProSeg foreground transcripts",
    ylabel="transcripts rejected as outliers",
)

source_counts = display_points[HYBRID_ASSIGNMENT_SOURCE_COLUMN].value_counts().sort_index()
source_counts.plot.bar(ax=axes[2], color="#54A24B")
axes[2].set(title="Final assignment provenance", xlabel="source", ylabel="transcripts")
axes[2].tick_params(axis="x", rotation=35)

fig.savefig(DIAGNOSTIC_PLOT_PATH, dpi=220, bbox_inches="tight")
plt.show()
display(diagnostics[
    [
        "proseg_foreground_transcripts",
        "retained_transcripts",
        "outlier_transcripts",
        "hybrid_assigned_transcripts",
        "fallback_reason",
        "area_ratio",
    ]
].sort_values(["outlier_transcripts", "area_ratio"], ascending=[False, False]).head(20))
print(f"Saved {DIAGNOSTIC_PLOT_PATH}")


## Outlier-parameter sweep

This sweep reruns the exact production geometry and assignment functions in memory for every `(outlier_neighbors, outlier_mad_multiplier)` combination. Concavity and smoothing are deliberately held fixed. Lower values of either outlier parameter make it easier to separate and reject small, weakly connected transcript groups. Each subplot uses the same cell colours and overlays the original Cellpose boundary, the condition-specific hybrid polygon, final assignments, and originally foreground ProSeg transcripts rejected by that condition.

In [ ]:
sweep_x_column = "observed_x" if "observed_x" in points_pdf else "x"
sweep_y_column = "observed_y" if "observed_y" in points_pdf else "y"
sweep_assignment_map = {
    int(cell): str(original)
    for cell, original in zip(
        selected_proseg_table.obs["cell"],
        selected_proseg_table.obs["original_cell_id"],
        strict=True,
    )
}
sweep_proseg_cells = (
    pd.to_numeric(points_pdf["assignment"], errors="coerce")
    .map(sweep_assignment_map)
    .astype("string")
)
foreground_valid = (
    ~points_pdf["background"].astype(bool)
    & sweep_proseg_cells.notna()
)
foreground_source = points_pdf.loc[
    foreground_valid, [sweep_x_column, sweep_y_column]
].copy()
foreground_source["_cell_id"] = sweep_proseg_cells.loc[foreground_valid].to_numpy()
foreground_coordinates = {
    str(cell_id): group[[sweep_x_column, sweep_y_column]].to_numpy(np.float64)
    for cell_id, group in foreground_source.groupby("_cell_id", sort=False)
}

sweep_cell_ids = sorted(cellpose_polygons, key=int)
sweep_results: dict[str, dict[str, object]] = {}
sweep_rows: list[dict[str, object]] = []

for neighbors, mad_multiplier in product(
    SWEEP_OUTLIER_NEIGHBORS, SWEEP_OUTLIER_MAD_MULTIPLIERS
):
    condition = f"k={neighbors}, MAD={mad_multiplier:g}"
    config_values = HYBRID_CONFIG.model_dump()
    config_values.update(
        outlier_neighbors=int(neighbors),
        outlier_mad_multiplier=float(mad_multiplier),
        concavity_ratio=float(SWEEP_CONCAVITY_RATIO),
        smoothing_scale=float(SWEEP_SMOOTHING_SCALE),
    )
    condition_config = ProsegHybridConfig.model_validate(config_values)

    cell_results = {
        cell_id: build_hybrid_cell_geometry(
            foreground_coordinates.get(
                cell_id, np.empty((0, 2), dtype=np.float64)
            ),
            cellpose_polygons[cell_id],
            condition_config,
        )
        for cell_id in sweep_cell_ids
    }
    polygons = [cell_results[cell_id].geometry for cell_id in sweep_cell_ids]
    augmented = assign_transcripts_to_hybrid_masks(
        points_pdf,
        polygons,
        sweep_cell_ids,
        sweep_assignment_map,
        x_col=sweep_x_column,
        y_col=sweep_y_column,
        assignment_col="assignment",
    )
    augmented["proseg_cell_id"] = sweep_proseg_cells.to_numpy()
    hybrid_frame = gpd.GeoDataFrame(
        {
            "cell_id": sweep_cell_ids,
            "geometry": polygons,
        },
        geometry="geometry",
        index=pd.Index(sweep_cell_ids, name="cell_id_index"),
    )

    compactness = np.asarray(
        [
            (4.0 * np.pi * geometry.area / (geometry.length**2))
            if geometry.length > 0.0 else np.nan
            for geometry in polygons
        ],
        dtype=np.float64,
    )
    area_ratios = np.asarray(
        [
            cell_results[cell_id].geometry.area / cellpose_polygons[cell_id].area
            for cell_id in sweep_cell_ids
        ],
        dtype=np.float64,
    )
    originally_foreground = (
        ~augmented["background"].astype(bool)
        & augmented["proseg_cell_id"].notna()
    )
    rejected_foreground = (
        originally_foreground
        & augmented[HYBRID_ASSIGNMENT_COLUMN].isna()
    )
    rescued_background = (
        augmented["background"].astype(bool)
        & augmented[HYBRID_ASSIGNMENT_COLUMN].notna()
    )
    row = {
        "condition": condition,
        "outlier_neighbors": int(neighbors),
        "outlier_mad_multiplier": float(mad_multiplier),
        "outlier_transcripts": int(
            sum(result.outlier_count for result in cell_results.values())
        ),
        "fallback_cells": int(
            sum(bool(result.fallback_reason) for result in cell_results.values())
        ),
        "hybrid_assigned_transcripts": int(
            augmented[HYBRID_ASSIGNMENT_COLUMN].notna().sum()
        ),
        "rejected_proseg_foreground": int(rejected_foreground.sum()),
        "rescued_proseg_background": int(rescued_background.sum()),
        "ambiguous_overlap": int(
            (augmented[HYBRID_ASSIGNMENT_SOURCE_COLUMN] == "ambiguous_overlap").sum()
        ),
        "median_area_ratio": float(np.nanmedian(area_ratios)),
        "median_compactness": float(np.nanmedian(compactness)),
    }
    sweep_rows.append(row)
    sweep_results[condition] = {
        "config": condition_config,
        "cells": cell_results,
        "shapes": hybrid_frame,
        "points": augmented,
        "summary": row,
    }

sweep_summary = pd.DataFrame(sweep_rows).set_index("condition")
sweep_summary.to_csv(SWEEP_SUMMARY_PATH)
display(sweep_summary)
print(f"Saved {SWEEP_SUMMARY_PATH}")


In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

sweep_x_min, sweep_y_min, sweep_x_max, sweep_y_max = SWEEP_DISPLAY_BBOX_UM
n_conditions = len(sweep_results)
n_columns = min(max(1, int(SWEEP_N_COLUMNS)), n_conditions)
n_rows = int(np.ceil(n_conditions / n_columns))
fig, axes = plt.subplots(
    n_rows, n_columns,
    figsize=(5.2 * n_columns, 5.0 * n_rows),
    sharex=True, sharey=True, constrained_layout=True, squeeze=False,
)
flat_axes = axes.ravel()

for ax, (condition, condition_result) in zip(
    flat_axes, sweep_results.items(), strict=False
):
    hybrid_frame = condition_result["shapes"]
    condition_points = condition_result["points"]
    summary = condition_result["summary"]
    in_sweep_view = (
        condition_points[sweep_x_column].between(sweep_x_min, sweep_x_max)
        & condition_points[sweep_y_column].between(sweep_y_min, sweep_y_max)
    )
    visible_points = condition_points.loc[in_sweep_view]

    plot_polygons_by_cell(
        ax,
        hybrid_frame,
        hybrid_frame["cell_id"],
        cell_colors,
        alpha=0.24,
    )
    # Original Cellpose boundary is always the same black dashed outline.
    cellpose_gdf.boundary.plot(
        ax=ax, color="black", linestyle="--", linewidth=0.65, alpha=0.8
    )
    plot_sweep_transcripts(ax, visible_points, cell_colors)
    ax.set_title(
        f"{condition}\n"
        f"outliers={summary['outlier_transcripts']:,}; "
        f"compactness={summary['median_compactness']:.3f}"
    )
    ax.set_xlim(sweep_x_min, sweep_x_max)
    ax.set_ylim(sweep_y_min, sweep_y_max)
    ax.set_aspect("equal")
    ax.set_xlabel("observed x (µm)")
    ax.set_ylabel("observed y (µm)")
    ax.grid(False)

for ax in flat_axes[n_conditions:]:
    ax.set_visible(False)

legend_handles = [
    Patch(facecolor="0.55", alpha=0.24, label="hybrid mask (cell colour)"),
    Line2D([0], [0], color="black", linestyle="--", linewidth=0.8, label="original Cellpose boundary"),
    Line2D([0], [0], marker="o", color="none", markerfacecolor="0.35", markersize=4, label="original ProSeg foreground, final assigned"),
    Line2D([0], [0], marker="+", color="0.5", linestyle="none", markersize=6, label="original ProSeg background/unassigned"),
    Line2D([0], [0], marker="x", color="black", linestyle="none", markersize=5, label="ProSeg foreground rejected"),
]
fig.legend(handles=legend_handles, loc="outside lower center", ncol=5, frameon=False)
fig.suptitle(
    "P7513 hybrid outlier sweep — lower k/MAD is harsher", fontsize=16
)
fig.savefig(SWEEP_PLOT_PATH, dpi=220, bbox_inches="tight")
plt.show()
print(f"Saved {SWEEP_PLOT_PATH}")


## Supported transcript envelope (experimental)

This alternative addresses the two remaining failure modes separately. Only ProSeg-foreground transcripts outside the original Cellpose polygon are allowed to extend a cell. Those external transcripts are placed in an adaptive neighbour graph and iteratively peeled until every survivor has at least `SUPPORTED_MIN_NEIGHBORS` other external neighbours. Groups not geometrically connected back to Cellpose after smoothing are discarded. The survivors are buffered into a rounded envelope, opened to remove thin outward structures, closed to bridge small gaps, and finally unioned with the exact original Cellpose polygon.

The support radius, envelope radius, opening, and closing distances are all multiples of the cell-specific robust neighbour scale—there is no fixed maximum-distance cutoff.

In [ ]:
def iterative_k_core_mask(
    points_xy: np.ndarray,
    *,
    radius: float,
    min_neighbors: int,
) -> np.ndarray:
    """Return nodes surviving iterative minimum-degree pruning."""
    points = np.asarray(points_xy, dtype=np.float64)
    n_points = len(points)
    if n_points == 0 or n_points <= min_neighbors or radius <= 0.0:
        return np.zeros(n_points, dtype=bool)

    tree = cKDTree(points)
    neighbor_sets = [
        set(int(value) for value in neighbors if int(value) != index)
        for index, neighbors in enumerate(tree.query_ball_point(points, r=radius))
    ]
    active = np.ones(n_points, dtype=bool)
    degrees = np.asarray([len(neighbors) for neighbors in neighbor_sets], dtype=np.int64)
    queue = deque(int(index) for index in np.flatnonzero(degrees < min_neighbors))
    while queue:
        index = queue.popleft()
        if not active[index]:
            continue
        active[index] = False
        for neighbor in neighbor_sets[index]:
            if not active[neighbor]:
                continue
            degrees[neighbor] -= 1
            if degrees[neighbor] < min_neighbors:
                queue.append(neighbor)
    return active


def polygonal_parts(geometry: object) -> list[object]:
    """Return polygonal components from a Shapely geometry."""
    if geometry.is_empty:
        return []
    if geometry.geom_type == "Polygon":
        return [geometry]
    if hasattr(geometry, "geoms"):
        parts: list[object] = []
        for part in geometry.geoms:
            parts.extend(polygonal_parts(part))
        return parts
    return []


def build_supported_envelope_geometry(
    points_xy: np.ndarray,
    cellpose_polygon: object,
    config: ProsegHybridConfig,
    *,
    extension_min_neighbors: int,
    extension_radius_scale: float,
    envelope_radius_scale: float,
    opening_scale: float,
    closing_scale: float,
) -> dict[str, object]:
    """Build an adaptive, rounded envelope from supported external transcripts."""
    points = np.asarray(points_xy, dtype=np.float64)
    empty_result = {
        "geometry": cellpose_polygon,
        "neighbor_scale": 0.0,
        "component_outliers": 0,
        "unsupported_external": 0,
        "unanchored_external": 0,
        "supported_external": 0,
        "fallback_reason": "low_transcript_count",
    }
    if len(points) < config.min_transcripts:
        return empty_result

    selection = select_bulk_transcripts(
        points,
        cellpose_polygon,
        neighbors=config.outlier_neighbors,
        mad_multiplier=config.outlier_mad_multiplier,
    )
    retained = selection.retained_points
    if len(retained) < config.min_transcripts:
        result = empty_result.copy()
        result.update(
            neighbor_scale=float(selection.neighbor_scale),
            component_outliers=int(selection.outlier_count),
            fallback_reason="low_retained_transcript_count",
        )
        return result

    outside = np.fromiter(
        (not cellpose_polygon.covers(Point(coordinate)) for coordinate in retained),
        dtype=bool,
        count=len(retained),
    )
    external = retained[outside]
    neighbor_scale = float(selection.neighbor_scale)
    if len(external) == 0 or neighbor_scale <= np.finfo(np.float64).eps:
        return {
            "geometry": cellpose_polygon,
            "neighbor_scale": neighbor_scale,
            "component_outliers": int(selection.outlier_count),
            "unsupported_external": int(len(external)),
            "unanchored_external": 0,
            "supported_external": 0,
            "fallback_reason": "",
        }

    support_radius = neighbor_scale * float(extension_radius_scale)
    support_mask = iterative_k_core_mask(
        external,
        radius=support_radius,
        min_neighbors=int(extension_min_neighbors),
    )
    supported = external[support_mask]
    unsupported_count = int(len(external) - len(supported))
    if len(supported) == 0:
        return {
            "geometry": cellpose_polygon,
            "neighbor_scale": neighbor_scale,
            "component_outliers": int(selection.outlier_count),
            "unsupported_external": unsupported_count,
            "unanchored_external": 0,
            "supported_external": 0,
            "fallback_reason": "",
        }

    envelope_radius = neighbor_scale * float(envelope_radius_scale)
    extension = MultiPoint(supported).buffer(envelope_radius, quad_segs=8)
    opening_distance = neighbor_scale * float(opening_scale)
    if opening_distance > 0.0 and not extension.is_empty:
        eroded = extension.buffer(-opening_distance, quad_segs=8)
        extension = (
            eroded.buffer(opening_distance, quad_segs=8)
            if not eroded.is_empty else eroded
        )

    combined = unary_union([cellpose_polygon, extension])
    closing_distance = neighbor_scale * float(closing_scale)
    if closing_distance > 0.0:
        closed = combined.buffer(closing_distance, quad_segs=8).buffer(
            -closing_distance, quad_segs=8
        )
        if not closed.is_empty and closed.is_valid:
            combined = closed

    # Disconnected transcript islands are not part of the cell, even if dense.
    anchored_parts = [
        part for part in polygonal_parts(combined)
        if part.intersects(cellpose_polygon)
    ]
    geometry = unary_union([cellpose_polygon, *anchored_parts])
    if not geometry.is_valid:
        geometry = geometry.buffer(0)
    supported_covered = int(
        sum(geometry.covers(Point(coordinate)) for coordinate in supported)
    )
    return {
        "geometry": geometry,
        "neighbor_scale": neighbor_scale,
        "component_outliers": int(selection.outlier_count),
        "unsupported_external": unsupported_count,
        "unanchored_external": int(len(supported) - supported_covered),
        "supported_external": supported_covered,
        "fallback_reason": "",
    }


In [ ]:
supported_config_values = HYBRID_CONFIG.model_dump()
supported_config_values.update(
    outlier_neighbors=SUPPORTED_BASE_OUTLIER_NEIGHBORS,
    outlier_mad_multiplier=SUPPORTED_BASE_OUTLIER_MAD_MULTIPLIER,
)
supported_base_config = ProsegHybridConfig.model_validate(supported_config_values)
supported_results: dict[str, dict[str, object]] = {}
supported_rows: list[dict[str, object]] = []

for min_neighbors, radius_scale in product(
    SUPPORTED_MIN_NEIGHBORS, SUPPORTED_RADIUS_SCALES
):
    condition = f"support≥{min_neighbors}, radius={radius_scale:g}×"
    cell_results = {
        cell_id: build_supported_envelope_geometry(
            foreground_coordinates.get(
                cell_id, np.empty((0, 2), dtype=np.float64)
            ),
            cellpose_polygons[cell_id],
            supported_base_config,
            extension_min_neighbors=int(min_neighbors),
            extension_radius_scale=float(radius_scale),
            envelope_radius_scale=float(SUPPORTED_ENVELOPE_RADIUS_SCALE),
            opening_scale=float(SUPPORTED_OPENING_SCALE),
            closing_scale=float(SUPPORTED_CLOSING_SCALE),
        )
        for cell_id in sweep_cell_ids
    }
    polygons = [cell_results[cell_id]["geometry"] for cell_id in sweep_cell_ids]
    augmented = assign_transcripts_to_hybrid_masks(
        points_pdf,
        polygons,
        sweep_cell_ids,
        sweep_assignment_map,
        x_col=sweep_x_column,
        y_col=sweep_y_column,
        assignment_col="assignment",
    )
    augmented["proseg_cell_id"] = sweep_proseg_cells.to_numpy()
    hybrid_frame = gpd.GeoDataFrame(
        {"cell_id": sweep_cell_ids, "geometry": polygons},
        geometry="geometry",
        index=pd.Index(sweep_cell_ids, name="cell_id_index"),
    )
    compactness = np.asarray(
        [
            (4.0 * np.pi * geometry.area / geometry.length**2)
            if geometry.length > 0.0 else np.nan
            for geometry in polygons
        ],
        dtype=np.float64,
    )
    area_ratios = np.asarray(
        [
            geometry.area / cellpose_polygons[cell_id].area
            for cell_id, geometry in zip(sweep_cell_ids, polygons, strict=True)
        ],
        dtype=np.float64,
    )
    originally_foreground = (
        ~augmented["background"].astype(bool)
        & augmented["proseg_cell_id"].notna()
    )
    rejected_foreground = (
        originally_foreground & augmented[HYBRID_ASSIGNMENT_COLUMN].isna()
    )
    rescued_background = (
        augmented["background"].astype(bool)
        & augmented[HYBRID_ASSIGNMENT_COLUMN].notna()
    )
    row = {
        "condition": condition,
        "extension_min_neighbors": int(min_neighbors),
        "extension_radius_scale": float(radius_scale),
        "component_outliers": int(
            sum(result["component_outliers"] for result in cell_results.values())
        ),
        "unsupported_external": int(
            sum(result["unsupported_external"] for result in cell_results.values())
        ),
        "unanchored_external": int(
            sum(result["unanchored_external"] for result in cell_results.values())
        ),
        "supported_external": int(
            sum(result["supported_external"] for result in cell_results.values())
        ),
        "fallback_cells": int(
            sum(bool(result["fallback_reason"]) for result in cell_results.values())
        ),
        "hybrid_assigned_transcripts": int(
            augmented[HYBRID_ASSIGNMENT_COLUMN].notna().sum()
        ),
        "rejected_proseg_foreground": int(rejected_foreground.sum()),
        "rescued_proseg_background": int(rescued_background.sum()),
        "ambiguous_overlap": int(
            (augmented[HYBRID_ASSIGNMENT_SOURCE_COLUMN] == "ambiguous_overlap").sum()
        ),
        "median_area_ratio": float(np.nanmedian(area_ratios)),
        "median_compactness": float(np.nanmedian(compactness)),
    }
    supported_rows.append(row)
    supported_results[condition] = {
        "cells": cell_results,
        "shapes": hybrid_frame,
        "points": augmented,
        "summary": row,
    }

supported_summary = pd.DataFrame(supported_rows).set_index("condition")
supported_summary.to_csv(SUPPORTED_SWEEP_SUMMARY_PATH)
display(supported_summary)
print(f"Saved {SUPPORTED_SWEEP_SUMMARY_PATH}")


In [ ]:
supported_x_min, supported_y_min, supported_x_max, supported_y_max = (
    SUPPORTED_DISPLAY_BBOX_UM
)
n_conditions = len(supported_results)
n_columns = min(max(1, int(SUPPORTED_N_COLUMNS)), n_conditions)
n_rows = int(np.ceil(n_conditions / n_columns))
fig, axes = plt.subplots(
    n_rows, n_columns,
    figsize=(5.6 * n_columns, 5.2 * n_rows),
    sharex=True, sharey=True, constrained_layout=True, squeeze=False,
)
flat_axes = axes.ravel()

for ax, (condition, condition_result) in zip(
    flat_axes, supported_results.items(), strict=False
):
    hybrid_frame = condition_result["shapes"]
    condition_points = condition_result["points"]
    summary = condition_result["summary"]
    visible = (
        condition_points[sweep_x_column].between(supported_x_min, supported_x_max)
        & condition_points[sweep_y_column].between(supported_y_min, supported_y_max)
    )
    visible_points = condition_points.loc[visible]
    plot_polygons_by_cell(
        ax, hybrid_frame, hybrid_frame["cell_id"], cell_colors, alpha=0.24
    )
    cellpose_gdf.boundary.plot(
        ax=ax, color="black", linestyle="--", linewidth=0.7, alpha=0.85
    )
    plot_sweep_transcripts(ax, visible_points, cell_colors)
    ax.set_title(
        f"{condition}\n"
        f"supported={summary['supported_external']:,}; "
        f"compactness={summary['median_compactness']:.3f}"
    )
    ax.set_xlim(supported_x_min, supported_x_max)
    ax.set_ylim(supported_y_min, supported_y_max)
    ax.set_aspect("equal")
    ax.set_xlabel("observed x (µm)")
    ax.set_ylabel("observed y (µm)")
    ax.grid(False)

for ax in flat_axes[n_conditions:]:
    ax.set_visible(False)
fig.legend(handles=legend_handles, loc="outside lower center", ncol=5, frameon=False)
fig.suptitle(
    "P7513 supported transcript envelope — k=2, MAD=2", fontsize=16
)
fig.savefig(SUPPORTED_SWEEP_PLOT_PATH, dpi=220, bbox_inches="tight")
plt.show()
print(f"Saved {SUPPORTED_SWEEP_PLOT_PATH}")


## Distance-weighted Cellpose-surface support (experimental)

This experiment replaces the fixed external-neighbour requirement with a soft score. For external transcript *i*, `score_i = (surface_reward + local_same_cell_support_i) × exp(-surface_distance_i / decay_length)`. The decay length is a fraction of the equivalent Cellpose radius, so it adapts to cell size. Local support is calculated exclusively from original ProSeg-foreground transcripts assigned to the same cell; ProSeg-background and unassigned transcripts have zero influence on geometry. They may be rescued by the final mask assignment, but those rescued calls are never fed back into expansion.

A nearby singleton can pass through its surface reward and create a rounded lobe. Farther transcript groups require progressively greater same-cell support, while disconnected islands are removed after envelope construction.

In [ ]:
def build_distance_weighted_envelope_geometry(
    points_xy: np.ndarray,
    cellpose_polygon: object,
    config: ProsegHybridConfig,
    *,
    decay_radius_fraction: float,
    surface_reward: float,
    local_support_radius_scale: float,
    min_effective_support: float,
    envelope_radius_scale: float,
    opening_scale: float,
    closing_scale: float,
) -> dict[str, object]:
    """Build a rounded extension from distance-weighted ProSeg support."""
    points = np.asarray(points_xy, dtype=np.float64)
    base_result: dict[str, object] = {
        "geometry": cellpose_polygon,
        "neighbor_scale": 0.0,
        "decay_length_um": 0.0,
        "component_outliers": 0,
        "candidate_external": 0,
        "unsupported_external": 0,
        "unanchored_external": 0,
        "supported_external": 0,
        "supported_distances_um": np.empty(0, dtype=np.float64),
        "supported_scores": np.empty(0, dtype=np.float64),
        "fallback_reason": "low_transcript_count",
    }
    if len(points) < config.min_transcripts:
        return base_result

    selection = select_bulk_transcripts(
        points,
        cellpose_polygon,
        neighbors=config.outlier_neighbors,
        mad_multiplier=config.outlier_mad_multiplier,
    )
    retained = selection.retained_points
    neighbor_scale = float(selection.neighbor_scale)
    if len(retained) < config.min_transcripts:
        result = base_result.copy()
        result.update(
            neighbor_scale=neighbor_scale,
            component_outliers=int(selection.outlier_count),
            fallback_reason="low_retained_transcript_count",
        )
        return result

    outside = np.fromiter(
        (not cellpose_polygon.covers(Point(coordinate)) for coordinate in retained),
        dtype=bool,
        count=len(retained),
    )
    external_indices = np.flatnonzero(outside)
    external = retained[external_indices]
    equivalent_radius = float(np.sqrt(cellpose_polygon.area / np.pi))
    decay_length = float(decay_radius_fraction) * equivalent_radius
    epsilon = float(np.finfo(np.float64).eps)
    if (
        len(external) == 0
        or neighbor_scale <= epsilon
        or decay_length <= epsilon
    ):
        result = base_result.copy()
        result.update(
            neighbor_scale=neighbor_scale,
            decay_length_um=decay_length,
            component_outliers=int(selection.outlier_count),
            candidate_external=int(len(external)),
            unsupported_external=int(len(external)),
            fallback_reason="",
        )
        return result

    local_radius = neighbor_scale * float(local_support_radius_scale)
    tree = cKDTree(retained)
    neighbor_lists = tree.query_ball_point(external, r=local_radius)
    local_support = np.zeros(len(external), dtype=np.float64)
    for external_index, (source_index, neighbors) in enumerate(
        zip(external_indices, neighbor_lists, strict=True)
    ):
        neighbor_indices = np.asarray(
            [int(index) for index in neighbors if int(index) != int(source_index)],
            dtype=np.int64,
        )
        if len(neighbor_indices) == 0:
            continue
        distances = np.linalg.norm(
            retained[neighbor_indices] - retained[source_index], axis=1
        )
        local_support[external_index] = float(
            np.exp(-0.5 * np.square(distances / local_radius)).sum()
        )

    surface_distances = np.fromiter(
        (cellpose_polygon.distance(Point(coordinate)) for coordinate in external),
        dtype=np.float64,
        count=len(external),
    )
    distance_weights = np.exp(-surface_distances / decay_length)
    effective_support = (float(surface_reward) + local_support) * distance_weights
    support_mask = effective_support >= float(min_effective_support)
    supported = external[support_mask]
    supported_distances = surface_distances[support_mask]
    supported_scores = effective_support[support_mask]
    unsupported_count = int(len(external) - len(supported))
    if len(supported) == 0:
        result = base_result.copy()
        result.update(
            neighbor_scale=neighbor_scale,
            decay_length_um=decay_length,
            component_outliers=int(selection.outlier_count),
            candidate_external=int(len(external)),
            unsupported_external=unsupported_count,
            fallback_reason="",
        )
        return result

    envelope_radius = neighbor_scale * float(envelope_radius_scale)
    extension = MultiPoint(supported).buffer(envelope_radius, quad_segs=8)
    opening_distance = neighbor_scale * float(opening_scale)
    if opening_distance > 0.0 and not extension.is_empty:
        eroded = extension.buffer(-opening_distance, quad_segs=8)
        extension = (
            eroded.buffer(opening_distance, quad_segs=8)
            if not eroded.is_empty else eroded
        )

    combined = unary_union([cellpose_polygon, extension])
    closing_distance = neighbor_scale * float(closing_scale)
    if closing_distance > 0.0:
        closed = combined.buffer(closing_distance, quad_segs=8).buffer(
            -closing_distance, quad_segs=8
        )
        if not closed.is_empty and closed.is_valid:
            combined = closed

    anchored_parts = [
        part for part in polygonal_parts(combined)
        if part.intersects(cellpose_polygon)
    ]
    geometry = unary_union([cellpose_polygon, *anchored_parts])
    if not geometry.is_valid:
        geometry = geometry.buffer(0)
    covered_mask = np.fromiter(
        (geometry.covers(Point(coordinate)) for coordinate in supported),
        dtype=bool,
        count=len(supported),
    )
    return {
        "geometry": geometry,
        "neighbor_scale": neighbor_scale,
        "decay_length_um": decay_length,
        "component_outliers": int(selection.outlier_count),
        "candidate_external": int(len(external)),
        "unsupported_external": unsupported_count,
        "unanchored_external": int(len(supported) - covered_mask.sum()),
        "supported_external": int(covered_mask.sum()),
        "supported_distances_um": supported_distances[covered_mask],
        "supported_scores": supported_scores[covered_mask],
        "fallback_reason": "",
    }


In [ ]:
distance_config_values = HYBRID_CONFIG.model_dump()
distance_config_values.update(
    outlier_neighbors=DISTANCE_BASE_OUTLIER_NEIGHBORS,
    outlier_mad_multiplier=DISTANCE_BASE_OUTLIER_MAD_MULTIPLIER,
)
distance_base_config = ProsegHybridConfig.model_validate(distance_config_values)
distance_results: dict[str, dict[str, object]] = {}
distance_rows: list[dict[str, object]] = []

for decay_fraction, surface_reward in product(
    DISTANCE_DECAY_RADIUS_FRACTIONS, DISTANCE_SURFACE_REWARDS
):
    condition = f"decay={decay_fraction:g}R, reward={surface_reward:g}"
    cell_results = {
        cell_id: build_distance_weighted_envelope_geometry(
            foreground_coordinates.get(
                cell_id, np.empty((0, 2), dtype=np.float64)
            ),
            cellpose_polygons[cell_id],
            distance_base_config,
            decay_radius_fraction=float(decay_fraction),
            surface_reward=float(surface_reward),
            local_support_radius_scale=float(DISTANCE_LOCAL_SUPPORT_RADIUS_SCALE),
            min_effective_support=float(DISTANCE_MIN_EFFECTIVE_SUPPORT),
            envelope_radius_scale=float(DISTANCE_ENVELOPE_RADIUS_SCALE),
            opening_scale=float(DISTANCE_OPENING_SCALE),
            closing_scale=float(DISTANCE_CLOSING_SCALE),
        )
        for cell_id in sweep_cell_ids
    }
    polygons = [cell_results[cell_id]["geometry"] for cell_id in sweep_cell_ids]
    augmented = assign_transcripts_to_hybrid_masks(
        points_pdf,
        polygons,
        sweep_cell_ids,
        sweep_assignment_map,
        x_col=sweep_x_column,
        y_col=sweep_y_column,
        assignment_col="assignment",
    )
    augmented["proseg_cell_id"] = sweep_proseg_cells.to_numpy()
    hybrid_frame = gpd.GeoDataFrame(
        {"cell_id": sweep_cell_ids, "geometry": polygons},
        geometry="geometry",
        index=pd.Index(sweep_cell_ids, name="cell_id_index"),
    )
    compactness = np.asarray(
        [
            (4.0 * np.pi * geometry.area / geometry.length**2)
            if geometry.length > 0.0 else np.nan
            for geometry in polygons
        ],
        dtype=np.float64,
    )
    area_ratios = np.asarray(
        [
            geometry.area / cellpose_polygons[cell_id].area
            for cell_id, geometry in zip(sweep_cell_ids, polygons, strict=True)
        ],
        dtype=np.float64,
    )
    supported_distance_parts = [
        result["supported_distances_um"]
        for result in cell_results.values()
        if len(result["supported_distances_um"]) > 0
    ]
    supported_distances = (
        np.concatenate(supported_distance_parts)
        if supported_distance_parts else np.empty(0, dtype=np.float64)
    )
    originally_foreground = (
        ~augmented["background"].astype(bool)
        & augmented["proseg_cell_id"].notna()
    )
    rejected_foreground = (
        originally_foreground & augmented[HYBRID_ASSIGNMENT_COLUMN].isna()
    )
    rescued_background = (
        augmented["background"].astype(bool)
        & augmented[HYBRID_ASSIGNMENT_COLUMN].notna()
    )
    row = {
        "condition": condition,
        "decay_radius_fraction": float(decay_fraction),
        "surface_reward": float(surface_reward),
        "component_outliers": int(
            sum(result["component_outliers"] for result in cell_results.values())
        ),
        "candidate_external": int(
            sum(result["candidate_external"] for result in cell_results.values())
        ),
        "unsupported_external": int(
            sum(result["unsupported_external"] for result in cell_results.values())
        ),
        "unanchored_external": int(
            sum(result["unanchored_external"] for result in cell_results.values())
        ),
        "supported_external": int(
            sum(result["supported_external"] for result in cell_results.values())
        ),
        "fallback_cells": int(
            sum(bool(result["fallback_reason"]) for result in cell_results.values())
        ),
        "hybrid_assigned_transcripts": int(
            augmented[HYBRID_ASSIGNMENT_COLUMN].notna().sum()
        ),
        "rejected_proseg_foreground": int(rejected_foreground.sum()),
        "rescued_proseg_background": int(rescued_background.sum()),
        "ambiguous_overlap": int(
            (augmented[HYBRID_ASSIGNMENT_SOURCE_COLUMN] == "ambiguous_overlap").sum()
        ),
        "median_supported_distance_um": (
            float(np.median(supported_distances)) if len(supported_distances) else np.nan
        ),
        "max_supported_distance_um": (
            float(np.max(supported_distances)) if len(supported_distances) else np.nan
        ),
        "median_area_ratio": float(np.nanmedian(area_ratios)),
        "median_compactness": float(np.nanmedian(compactness)),
    }
    distance_rows.append(row)
    distance_results[condition] = {
        "cells": cell_results,
        "shapes": hybrid_frame,
        "points": augmented,
        "summary": row,
    }

distance_summary = pd.DataFrame(distance_rows).set_index("condition")
distance_summary.to_csv(DISTANCE_SWEEP_SUMMARY_PATH)
display(distance_summary)
print(f"Saved {DISTANCE_SWEEP_SUMMARY_PATH}")


In [ ]:
distance_x_min, distance_y_min, distance_x_max, distance_y_max = (
    DISTANCE_DISPLAY_BBOX_UM
)
n_conditions = len(distance_results)
n_columns = min(max(1, int(DISTANCE_N_COLUMNS)), n_conditions)
n_rows = int(np.ceil(n_conditions / n_columns))
fig, axes = plt.subplots(
    n_rows, n_columns,
    figsize=(5.6 * n_columns, 5.2 * n_rows),
    sharex=True, sharey=True, constrained_layout=True, squeeze=False,
)
flat_axes = axes.ravel()

for ax, (condition, condition_result) in zip(
    flat_axes, distance_results.items(), strict=False
):
    hybrid_frame = condition_result["shapes"]
    condition_points = condition_result["points"]
    summary = condition_result["summary"]
    visible = (
        condition_points[sweep_x_column].between(distance_x_min, distance_x_max)
        & condition_points[sweep_y_column].between(distance_y_min, distance_y_max)
    )
    visible_points = condition_points.loc[visible]
    plot_polygons_by_cell(
        ax, hybrid_frame, hybrid_frame["cell_id"], cell_colors, alpha=0.24
    )
    cellpose_gdf.boundary.plot(
        ax=ax, color="black", linestyle="--", linewidth=0.7, alpha=0.85
    )
    plot_sweep_transcripts(ax, visible_points, cell_colors)
    ax.set_title(
        f"{condition}\n"
        f"supported={summary['supported_external']:,}; "
        f"compactness={summary['median_compactness']:.3f}"
    )
    ax.set_xlim(distance_x_min, distance_x_max)
    ax.set_ylim(distance_y_min, distance_y_max)
    ax.set_aspect("equal")
    ax.set_xlabel("observed x (µm)")
    ax.set_ylabel("observed y (µm)")
    ax.grid(False)

for ax in flat_axes[n_conditions:]:
    ax.set_visible(False)
fig.legend(handles=legend_handles, loc="outside lower center", ncol=5, frameon=False)
fig.suptitle(
    "P7513 distance-weighted Cellpose-surface support — k=2, MAD=2",
    fontsize=16,
)
fig.savefig(DISTANCE_SWEEP_PLOT_PATH, dpi=220, bbox_inches="tight")
plt.show()
print(f"Saved {DISTANCE_SWEEP_PLOT_PATH}")


## Tight-boundary geometry sweep at decay=0.5R, reward=2

This focused sweep holds transcript selection fixed and varies only how tightly the rounded envelope follows accepted transcripts. It uses `opening=0.25`, envelope radius scales `{0.35, 0.5, 0.75}`, and closing scales `{0.15, 0.3, 0.5}`.

In [ ]:
TIGHT_SWEEP_DECAY_RADIUS_FRACTION = 0.5
TIGHT_SWEEP_SURFACE_REWARD = 2.0
TIGHT_SWEEP_OPENING_SCALE = 0.25
TIGHT_SWEEP_ENVELOPE_RADIUS_SCALES = (0.35, 0.5, 0.75)
TIGHT_SWEEP_CLOSING_SCALES = (0.15, 0.3, 0.5)
TIGHT_SWEEP_DISPLAY_BBOX_UM = DISTANCE_DISPLAY_BBOX_UM
TIGHT_SWEEP_PLOT_PATH = OUTPUT_DIR / "P7513_distance_weighted_tight_geometry_sweep.png"
TIGHT_SWEEP_SUMMARY_PATH = OUTPUT_DIR / "P7513_distance_weighted_tight_geometry_sweep.csv"

tight_results: dict[str, dict[str, object]] = {}
tight_rows: list[dict[str, object]] = []
for envelope_scale, closing_scale in product(
    TIGHT_SWEEP_ENVELOPE_RADIUS_SCALES, TIGHT_SWEEP_CLOSING_SCALES
):
    condition = f"envelope={envelope_scale:g}×, close={closing_scale:g}×"
    cell_results = {
        cell_id: build_distance_weighted_envelope_geometry(
            foreground_coordinates.get(
                cell_id, np.empty((0, 2), dtype=np.float64)
            ),
            cellpose_polygons[cell_id],
            distance_base_config,
            decay_radius_fraction=TIGHT_SWEEP_DECAY_RADIUS_FRACTION,
            surface_reward=TIGHT_SWEEP_SURFACE_REWARD,
            local_support_radius_scale=float(DISTANCE_LOCAL_SUPPORT_RADIUS_SCALE),
            min_effective_support=float(DISTANCE_MIN_EFFECTIVE_SUPPORT),
            envelope_radius_scale=float(envelope_scale),
            opening_scale=TIGHT_SWEEP_OPENING_SCALE,
            closing_scale=float(closing_scale),
        )
        for cell_id in sweep_cell_ids
    }
    polygons = [cell_results[cell_id]["geometry"] for cell_id in sweep_cell_ids]
    augmented = assign_transcripts_to_hybrid_masks(
        points_pdf,
        polygons,
        sweep_cell_ids,
        sweep_assignment_map,
        x_col=sweep_x_column,
        y_col=sweep_y_column,
        assignment_col="assignment",
    )
    augmented["proseg_cell_id"] = sweep_proseg_cells.to_numpy()
    hybrid_frame = gpd.GeoDataFrame(
        {"cell_id": sweep_cell_ids, "geometry": polygons},
        geometry="geometry",
        index=pd.Index(sweep_cell_ids, name="cell_id_index"),
    )
    compactness = np.asarray(
        [
            (4.0 * np.pi * geometry.area / geometry.length**2)
            if geometry.length > 0.0 else np.nan
            for geometry in polygons
        ],
        dtype=np.float64,
    )
    area_ratios = np.asarray(
        [
            geometry.area / cellpose_polygons[cell_id].area
            for cell_id, geometry in zip(sweep_cell_ids, polygons, strict=True)
        ],
        dtype=np.float64,
    )
    originally_foreground = (
        ~augmented["background"].astype(bool)
        & augmented["proseg_cell_id"].notna()
    )
    rejected_foreground = (
        originally_foreground & augmented[HYBRID_ASSIGNMENT_COLUMN].isna()
    )
    rescued_background = (
        augmented["background"].astype(bool)
        & augmented[HYBRID_ASSIGNMENT_COLUMN].notna()
    )
    row = {
        "condition": condition,
        "envelope_radius_scale": float(envelope_scale),
        "closing_scale": float(closing_scale),
        "supported_external": int(
            sum(result["supported_external"] for result in cell_results.values())
        ),
        "unanchored_external": int(
            sum(result["unanchored_external"] for result in cell_results.values())
        ),
        "hybrid_assigned_transcripts": int(
            augmented[HYBRID_ASSIGNMENT_COLUMN].notna().sum()
        ),
        "rejected_proseg_foreground": int(rejected_foreground.sum()),
        "rescued_proseg_background": int(rescued_background.sum()),
        "ambiguous_overlap": int(
            (augmented[HYBRID_ASSIGNMENT_SOURCE_COLUMN] == "ambiguous_overlap").sum()
        ),
        "median_area_ratio": float(np.nanmedian(area_ratios)),
        "median_compactness": float(np.nanmedian(compactness)),
    }
    tight_rows.append(row)
    tight_results[condition] = {
        "shapes": hybrid_frame,
        "points": augmented,
        "summary": row,
    }

tight_summary = pd.DataFrame(tight_rows).set_index("condition")
tight_summary.to_csv(TIGHT_SWEEP_SUMMARY_PATH)
display(tight_summary)

tight_x_min, tight_y_min, tight_x_max, tight_y_max = TIGHT_SWEEP_DISPLAY_BBOX_UM
fig, axes = plt.subplots(
    3, 3, figsize=(16.8, 15.6), sharex=True, sharey=True,
    constrained_layout=True, squeeze=False,
)
for ax, (condition, condition_result) in zip(
    axes.ravel(), tight_results.items(), strict=True
):
    hybrid_frame = condition_result["shapes"]
    condition_points = condition_result["points"]
    summary = condition_result["summary"]
    visible = (
        condition_points[sweep_x_column].between(tight_x_min, tight_x_max)
        & condition_points[sweep_y_column].between(tight_y_min, tight_y_max)
    )
    visible_points = condition_points.loc[visible]
    plot_polygons_by_cell(
        ax, hybrid_frame, hybrid_frame["cell_id"], cell_colors, alpha=0.24
    )
    cellpose_gdf.boundary.plot(
        ax=ax, color="black", linestyle="--", linewidth=0.7, alpha=0.85
    )
    plot_sweep_transcripts(ax, visible_points, cell_colors)
    ax.set_title(
        f"{condition}\n"
        f"area={summary['median_area_ratio']:.3f}×; "
        f"compactness={summary['median_compactness']:.3f}"
    )
    ax.set_xlim(tight_x_min, tight_x_max)
    ax.set_ylim(tight_y_min, tight_y_max)
    ax.set_aspect("equal")
    ax.set_xlabel("observed x (µm)")
    ax.set_ylabel("observed y (µm)")
    ax.grid(False)

fig.legend(handles=legend_handles, loc="outside lower center", ncol=5, frameon=False)
fig.suptitle(
    "P7513 tight-boundary sweep — decay=0.5R, surface reward=2", fontsize=16
)
fig.savefig(TIGHT_SWEEP_PLOT_PATH, dpi=220, bbox_inches="tight")
plt.show()
print(f"Saved {TIGHT_SWEEP_PLOT_PATH}")
print(f"Saved {TIGHT_SWEEP_SUMMARY_PATH}")


## Local convex expansions (experimental)

This experiment keeps the Cellpose polygon as an exact lower bound and adds a separate local convex wedge for each supported external transcript group. Only retained ProSeg foreground transcripts can drive geometry. External groups must form a neighbour chain back to the Cellpose surface; groups contain at least three transcripts, while retained transcripts in the near-surface safe zone are accepted even as singletons. Each wedge attaches to a short Cellpose boundary arc, is rounded outwards, and is clipped to a fixed `1.0R` Cellpose dilation. Accepted driver transcripts are then added back with a numerical containment buffer and asserted to be covered by the final mask.

The explicit choices `minimum group = 3` and `maximum expansion = 1.0R` stay fixed. The 24 conditions sweep chain radius (2), near-surface safe-zone width (3), attachment-arc width (2), and rounding radius (2). Background/unassigned transcripts never enter the geometry calculation, but the normal overlap-aware reassignment can rescue them afterward.

In [ ]:
from shapely.ops import nearest_points


def graph_components(neighbor_sets: list[set[int]]) -> list[np.ndarray]:
    """Return connected components for an undirected adjacency list."""
    unseen = set(range(len(neighbor_sets)))
    components: list[np.ndarray] = []
    while unseen:
        seed = unseen.pop()
        component = [seed]
        queue = deque([seed])
        while queue:
            index = queue.popleft()
            discovered = neighbor_sets[index].intersection(unseen)
            if not discovered:
                continue
            unseen.difference_update(discovered)
            component.extend(discovered)
            queue.extend(discovered)
        components.append(np.asarray(component, dtype=np.int64))
    return components


def subset_graph_components(
    neighbor_sets: list[set[int]],
    selected_indices: np.ndarray,
) -> list[np.ndarray]:
    """Split selected graph nodes without allowing excluded nodes to bridge them."""
    selected = set(int(index) for index in selected_indices)
    groups: list[np.ndarray] = []
    while selected:
        seed = selected.pop()
        group = [seed]
        queue = deque([seed])
        while queue:
            index = queue.popleft()
            discovered = neighbor_sets[index].intersection(selected)
            if not discovered:
                continue
            selected.difference_update(discovered)
            group.extend(discovered)
            queue.extend(discovered)
        groups.append(np.asarray(group, dtype=np.int64))
    return groups


def attachment_arc_coordinates(
    cellpose_polygon: object,
    group_points: np.ndarray,
    *,
    neighbor_scale: float,
    width_scale: float,
    samples: int = 9,
) -> np.ndarray:
    """Sample a local boundary arc centred on the group's nearest surface."""
    group_geometry = MultiPoint(group_points)
    cellpose_parts = polygonal_parts(cellpose_polygon)
    if not cellpose_parts:
        return np.empty((0, 2), dtype=np.float64)
    ring = min(
        (part.exterior for part in cellpose_parts),
        key=lambda candidate: candidate.distance(group_geometry),
    )
    ring_length = float(ring.length)
    if ring_length <= np.finfo(np.float64).eps:
        return np.empty((0, 2), dtype=np.float64)

    nearest_surface = nearest_points(group_geometry, ring)[1]
    centre_distance = float(ring.project(nearest_surface))
    tangent_step = min(
        max(float(neighbor_scale), 0.01 * ring_length),
        0.1 * ring_length,
    )
    before = np.asarray(
        ring.interpolate((centre_distance - tangent_step) % ring_length).coords[0],
        dtype=np.float64,
    )
    after = np.asarray(
        ring.interpolate((centre_distance + tangent_step) % ring_length).coords[0],
        dtype=np.float64,
    )
    tangent = after - before
    tangent_norm = float(np.linalg.norm(tangent))
    if tangent_norm <= np.finfo(np.float64).eps:
        tangent = np.asarray([1.0, 0.0], dtype=np.float64)
    else:
        tangent /= tangent_norm

    projections = (group_points - np.asarray(nearest_surface.coords[0])) @ tangent
    tangential_span = float(np.ptp(projections)) if len(projections) > 1 else 0.0
    group_width = max(2.0 * float(neighbor_scale), tangential_span + 2.0 * float(neighbor_scale))
    arc_width = min(
        0.45 * ring_length,
        max(float(neighbor_scale), float(width_scale) * group_width),
    )
    offsets = np.linspace(-0.5 * arc_width, 0.5 * arc_width, int(samples))
    return np.asarray(
        [
            ring.interpolate((centre_distance + offset) % ring_length).coords[0]
            for offset in offsets
        ],
        dtype=np.float64,
    )


def build_local_convex_expansion_geometry(
    points_xy: np.ndarray,
    cellpose_polygon: object,
    config: ProsegHybridConfig,
    *,
    minimum_external_group: int,
    chain_radius_scale: float,
    near_surface_radius_fraction: float,
    maximum_expansion_radius_fraction: float,
    attachment_arc_width_scale: float,
    rounding_radius_fraction: float,
) -> dict[str, object]:
    """Add capped, rounded local convex wedges supported by anchored chains."""
    points = np.asarray(points_xy, dtype=np.float64)
    base_result: dict[str, object] = {
        "geometry": cellpose_polygon,
        "neighbor_scale": 0.0,
        "component_outliers": 0,
        "candidate_external": 0,
        "cap_rejected_external": 0,
        "unsupported_external": 0,
        "near_surface_accepted": 0,
        "chain_accepted": 0,
        "supported_external": 0,
        "unanchored_external": 0,
        "accepted_groups": 0,
        "supported_coordinates": np.empty((0, 2), dtype=np.float64),
        "fallback_reason": "low_transcript_count",
    }
    if len(points) < config.min_transcripts:
        return base_result

    selection = select_bulk_transcripts(
        points,
        cellpose_polygon,
        neighbors=config.outlier_neighbors,
        mad_multiplier=config.outlier_mad_multiplier,
    )
    retained = selection.retained_points
    neighbor_scale = float(selection.neighbor_scale)
    if len(retained) < config.min_transcripts:
        result = base_result.copy()
        result.update(
            neighbor_scale=neighbor_scale,
            component_outliers=int(selection.outlier_count),
            fallback_reason="low_retained_transcript_count",
        )
        return result

    outside = np.fromiter(
        (not cellpose_polygon.covers(Point(coordinate)) for coordinate in retained),
        dtype=bool,
        count=len(retained),
    )
    external = retained[outside]
    equivalent_radius = float(np.sqrt(cellpose_polygon.area / np.pi))
    cap_distance = float(maximum_expansion_radius_fraction) * equivalent_radius
    near_distance = float(near_surface_radius_fraction) * equivalent_radius
    epsilon = max(np.finfo(np.float64).eps, equivalent_radius * 1.0e-7)
    containment_epsilon = max(1.0e-4, equivalent_radius * 1.0e-5)
    if len(external) == 0 or neighbor_scale <= epsilon or cap_distance <= epsilon:
        result = base_result.copy()
        result.update(
            neighbor_scale=neighbor_scale,
            component_outliers=int(selection.outlier_count),
            candidate_external=int(len(external)),
            unsupported_external=int(len(external)),
            fallback_reason="",
        )
        return result

    external_distances = np.fromiter(
        (cellpose_polygon.distance(Point(coordinate)) for coordinate in external),
        dtype=np.float64,
        count=len(external),
    )
    within_cap = external_distances <= cap_distance
    candidates = external[within_cap]
    candidate_distances = external_distances[within_cap]
    cap_rejected = int((~within_cap).sum())
    if len(candidates) == 0:
        result = base_result.copy()
        result.update(
            neighbor_scale=neighbor_scale,
            component_outliers=int(selection.outlier_count),
            candidate_external=int(len(external)),
            cap_rejected_external=cap_rejected,
            unsupported_external=int(len(external)),
            fallback_reason="",
        )
        return result

    chain_radius = float(chain_radius_scale) * neighbor_scale
    candidate_tree = cKDTree(candidates)
    neighbor_sets = [
        set(int(value) for value in neighbors if int(value) != index)
        for index, neighbors in enumerate(
            candidate_tree.query_ball_point(candidates, r=chain_radius)
        )
    ]
    components = graph_components(neighbor_sets)
    near_surface = candidate_distances <= near_distance + epsilon
    chain_anchors = candidate_distances <= chain_radius + epsilon
    accepted = near_surface.copy()
    for component in components:
        if (
            len(component) >= int(minimum_external_group)
            and bool(chain_anchors[component].any())
        ):
            accepted[component] = True

    accepted_indices = np.flatnonzero(accepted)
    if len(accepted_indices) == 0:
        result = base_result.copy()
        result.update(
            neighbor_scale=neighbor_scale,
            component_outliers=int(selection.outlier_count),
            candidate_external=int(len(external)),
            cap_rejected_external=cap_rejected,
            unsupported_external=int(len(external)),
            fallback_reason="",
        )
        return result

    accepted_groups = subset_graph_components(neighbor_sets, accepted_indices)
    # Shapely approximates circular buffers with inscribed chords. Inflate
    # by the known chord factor so the polygon covers the analytical cap;
    # at 64 segments/quadrant the maximum overshoot is sub-nanometre.
    cap_quad_segs = 64
    cap_buffer_distance = (cap_distance + containment_epsilon) / np.cos(
        np.pi / (4.0 * cap_quad_segs)
    )
    cap_region = cellpose_polygon.buffer(
        cap_buffer_distance, quad_segs=cap_quad_segs
    )
    rounding_radius = float(rounding_radius_fraction) * equivalent_radius
    extensions: list[object] = []
    for group_indices in accepted_groups:
        group_points = candidates[group_indices]
        arc_points = attachment_arc_coordinates(
            cellpose_polygon,
            group_points,
            neighbor_scale=neighbor_scale,
            width_scale=float(attachment_arc_width_scale),
        )
        if len(arc_points) == 0:
            continue
        local_hull = MultiPoint(np.vstack([group_points, arc_points])).convex_hull
        rounded_hull = (
            local_hull.buffer(rounding_radius, quad_segs=12)
            if rounding_radius > 0.0 else local_hull
        )
        clipped_hull = rounded_hull.intersection(cap_region)
        if not clipped_hull.is_empty:
            extensions.append(clipped_hull)

    supported = candidates[accepted_indices]
    containment_support = MultiPoint(supported).buffer(
        containment_epsilon, quad_segs=4
    )
    containment_support = containment_support.intersection(cap_region)
    geometry = unary_union(
        [cellpose_polygon, containment_support, *extensions]
    )
    if not geometry.is_valid:
        geometry = geometry.buffer(0)

    covered_mask = np.fromiter(
        (geometry.covers(Point(coordinate)) for coordinate in supported),
        dtype=bool,
        count=len(supported),
    )
    if not bool(covered_mask.all()):
        raise AssertionError(
            "Local convex smoothing excluded a geometry-driving transcript"
        )
    if cellpose_polygon.difference(geometry).area > max(1.0e-8, cellpose_polygon.area * 1.0e-10):
        raise AssertionError("Local convex geometry no longer covers Cellpose")

    chain_only = accepted & ~near_surface
    return {
        "geometry": geometry,
        "neighbor_scale": neighbor_scale,
        "component_outliers": int(selection.outlier_count),
        "candidate_external": int(len(external)),
        "cap_rejected_external": cap_rejected,
        "unsupported_external": int(len(external) - len(supported)),
        "near_surface_accepted": int((accepted & near_surface).sum()),
        "chain_accepted": int(chain_only.sum()),
        "supported_external": int(len(supported)),
        "unanchored_external": 0,
        "accepted_groups": int(len(accepted_groups)),
        "supported_coordinates": supported,
        "fallback_reason": "",
    }


In [ ]:
CONVEX_MINIMUM_EXTERNAL_GROUP = 3
CONVEX_MAXIMUM_EXPANSION_RADIUS_FRACTION = 1.0
CONVEX_CHAIN_RADIUS_SCALES = (1.5, 2.0)
CONVEX_NEAR_SURFACE_RADIUS_FRACTIONS = (0.15, 0.2, 0.25)
CONVEX_ATTACHMENT_ARC_WIDTH_SCALES = (0.5, 1.0)
CONVEX_ROUNDING_RADIUS_FRACTIONS = (0.15, 0.25)
CONVEX_DISPLAY_BBOX_UM = DISTANCE_DISPLAY_BBOX_UM
CONVEX_SWEEP_SUMMARY_PATH = OUTPUT_DIR / "P7513_local_convex_expansion_sweep.csv"
CONVEX_SWEEP_PLOT_STEM = OUTPUT_DIR / "P7513_local_convex_expansion_sweep"

convex_config_values = HYBRID_CONFIG.model_dump()
convex_config_values.update(
    outlier_neighbors=2,
    outlier_mad_multiplier=2.0,
)
convex_base_config = ProsegHybridConfig.model_validate(convex_config_values)
convex_results: dict[str, dict[str, object]] = {}
convex_rows: list[dict[str, object]] = []

convex_conditions = list(
    product(
        CONVEX_CHAIN_RADIUS_SCALES,
        CONVEX_NEAR_SURFACE_RADIUS_FRACTIONS,
        CONVEX_ATTACHMENT_ARC_WIDTH_SCALES,
        CONVEX_ROUNDING_RADIUS_FRACTIONS,
    )
)
assert len(convex_conditions) == 24

for chain_scale, near_fraction, arc_scale, rounding_fraction in convex_conditions:
    condition = (
        f"chain={chain_scale:g}×, near={near_fraction:g}R, "
        f"arc={arc_scale:g}×, round={rounding_fraction:g}R"
    )
    cell_results = {
        cell_id: build_local_convex_expansion_geometry(
            foreground_coordinates.get(
                cell_id, np.empty((0, 2), dtype=np.float64)
            ),
            cellpose_polygons[cell_id],
            convex_base_config,
            minimum_external_group=CONVEX_MINIMUM_EXTERNAL_GROUP,
            chain_radius_scale=float(chain_scale),
            near_surface_radius_fraction=float(near_fraction),
            maximum_expansion_radius_fraction=(
                CONVEX_MAXIMUM_EXPANSION_RADIUS_FRACTION
            ),
            attachment_arc_width_scale=float(arc_scale),
            rounding_radius_fraction=float(rounding_fraction),
        )
        for cell_id in sweep_cell_ids
    }
    if any(result["unanchored_external"] for result in cell_results.values()):
        raise AssertionError(f"Strict transcript containment failed for {condition}")

    polygons = [cell_results[cell_id]["geometry"] for cell_id in sweep_cell_ids]
    augmented = assign_transcripts_to_hybrid_masks(
        points_pdf,
        polygons,
        sweep_cell_ids,
        sweep_assignment_map,
        x_col=sweep_x_column,
        y_col=sweep_y_column,
        assignment_col="assignment",
    )
    augmented["proseg_cell_id"] = sweep_proseg_cells.to_numpy()
    hybrid_frame = gpd.GeoDataFrame(
        {"cell_id": sweep_cell_ids, "geometry": polygons},
        geometry="geometry",
        index=pd.Index(sweep_cell_ids, name="cell_id_index"),
    )
    compactness = np.asarray(
        [
            (4.0 * np.pi * geometry.area / geometry.length**2)
            if geometry.length > 0.0 else np.nan
            for geometry in polygons
        ],
        dtype=np.float64,
    )
    area_ratios = np.asarray(
        [
            geometry.area / cellpose_polygons[cell_id].area
            for cell_id, geometry in zip(sweep_cell_ids, polygons, strict=True)
        ],
        dtype=np.float64,
    )
    originally_foreground = (
        ~augmented["background"].astype(bool)
        & augmented["proseg_cell_id"].notna()
    )
    rejected_foreground = (
        originally_foreground & augmented[HYBRID_ASSIGNMENT_COLUMN].isna()
    )
    rescued_background = (
        augmented["background"].astype(bool)
        & augmented[HYBRID_ASSIGNMENT_COLUMN].notna()
    )
    row = {
        "condition": condition,
        "minimum_external_group": CONVEX_MINIMUM_EXTERNAL_GROUP,
        "maximum_expansion_radius_fraction": (
            CONVEX_MAXIMUM_EXPANSION_RADIUS_FRACTION
        ),
        "chain_radius_scale": float(chain_scale),
        "near_surface_radius_fraction": float(near_fraction),
        "attachment_arc_width_scale": float(arc_scale),
        "rounding_radius_fraction": float(rounding_fraction),
        "component_outliers": int(
            sum(result["component_outliers"] for result in cell_results.values())
        ),
        "candidate_external": int(
            sum(result["candidate_external"] for result in cell_results.values())
        ),
        "cap_rejected_external": int(
            sum(result["cap_rejected_external"] for result in cell_results.values())
        ),
        "unsupported_external": int(
            sum(result["unsupported_external"] for result in cell_results.values())
        ),
        "near_surface_accepted": int(
            sum(result["near_surface_accepted"] for result in cell_results.values())
        ),
        "chain_accepted": int(
            sum(result["chain_accepted"] for result in cell_results.values())
        ),
        "supported_external": int(
            sum(result["supported_external"] for result in cell_results.values())
        ),
        "accepted_groups": int(
            sum(result["accepted_groups"] for result in cell_results.values())
        ),
        "fallback_cells": int(
            sum(bool(result["fallback_reason"]) for result in cell_results.values())
        ),
        "hybrid_assigned_transcripts": int(
            augmented[HYBRID_ASSIGNMENT_COLUMN].notna().sum()
        ),
        "rejected_proseg_foreground": int(rejected_foreground.sum()),
        "rescued_proseg_background": int(rescued_background.sum()),
        "ambiguous_overlap": int(
            (augmented[HYBRID_ASSIGNMENT_SOURCE_COLUMN] == "ambiguous_overlap").sum()
        ),
        "median_area_ratio": float(np.nanmedian(area_ratios)),
        "median_compactness": float(np.nanmedian(compactness)),
    }
    convex_rows.append(row)
    convex_results[condition] = {
        "cells": cell_results,
        "shapes": hybrid_frame,
        "points": augmented,
        "summary": row,
    }

convex_summary = pd.DataFrame(convex_rows).set_index("condition")
convex_summary.to_csv(CONVEX_SWEEP_SUMMARY_PATH)
display(convex_summary)
print(f"Saved {CONVEX_SWEEP_SUMMARY_PATH}")


In [ ]:
convex_x_min, convex_y_min, convex_x_max, convex_y_max = (
    CONVEX_DISPLAY_BBOX_UM
)
CONVEX_CONDITIONS_PER_PAGE = 8
CONVEX_N_COLUMNS = 4
convex_plot_paths: list[Path] = []
convex_items = list(convex_results.items())

for page_start in range(0, len(convex_items), CONVEX_CONDITIONS_PER_PAGE):
    page_items = convex_items[
        page_start:page_start + CONVEX_CONDITIONS_PER_PAGE
    ]
    page_number = page_start // CONVEX_CONDITIONS_PER_PAGE + 1
    n_rows = int(np.ceil(len(page_items) / CONVEX_N_COLUMNS))
    fig, axes = plt.subplots(
        n_rows, CONVEX_N_COLUMNS,
        figsize=(5.5 * CONVEX_N_COLUMNS, 5.2 * n_rows),
        sharex=True, sharey=True, constrained_layout=True, squeeze=False,
    )
    flat_axes = axes.ravel()
    for ax, (condition, condition_result) in zip(
        flat_axes, page_items, strict=False
    ):
        hybrid_frame = condition_result["shapes"]
        condition_points = condition_result["points"]
        summary = condition_result["summary"]
        visible = (
            condition_points[sweep_x_column].between(convex_x_min, convex_x_max)
            & condition_points[sweep_y_column].between(convex_y_min, convex_y_max)
        )
        visible_points = condition_points.loc[visible]
        plot_polygons_by_cell(
            ax, hybrid_frame, hybrid_frame["cell_id"], cell_colors, alpha=0.24
        )
        cellpose_gdf.boundary.plot(
            ax=ax, color="black", linestyle="--", linewidth=0.7, alpha=0.85
        )
        plot_sweep_transcripts(ax, visible_points, cell_colors)
        ax.set_title(
            f"{condition}\n"
            f"supported={summary['supported_external']:,}; "
            f"area={summary['median_area_ratio']:.3f}×"
        )
        ax.set_xlim(convex_x_min, convex_x_max)
        ax.set_ylim(convex_y_min, convex_y_max)
        ax.set_aspect("equal")
        ax.set_xlabel("observed x (µm)")
        ax.set_ylabel("observed y (µm)")
        ax.grid(False)

    for ax in flat_axes[len(page_items):]:
        ax.set_visible(False)
    fig.legend(
        handles=legend_handles, loc="outside lower center", ncol=5, frameon=False
    )
    fig.suptitle(
        "P7513 local convex expansions — min group=3, cap=1.0R "
        f"(page {page_number}/3)",
        fontsize=16,
    )
    plot_path = Path(f"{CONVEX_SWEEP_PLOT_STEM}_page_{page_number}.png")
    fig.savefig(plot_path, dpi=200, bbox_inches="tight")
    convex_plot_paths.append(plot_path)
    plt.show()
    print(f"Saved {plot_path}")


## Larger inclusive-convex viewer export

These cells run the selected inclusive condition (`chain=2×`, `near=0.25R`, `arc=0.5×`, `round=0.15R`) on a region five times the width and height of `REGION_BBOX_UM`. The enlarged box retains the same centre and is written to a separate SpatialData Zarr; it does not overwrite the small test store or any production branch.

All transcripts assigned by ProSeg to the selected cells are considered during geometry construction, even if remote. The viewer point layer is cropped to the large box plus `MASK_CONTEXT_UM`, preventing rejected distant outliers from stretching the displayed extent.

In [ ]:
# -------------------- Large inclusive-run settings --------------------
LARGE_SUBSET_SCALE = 5.0
small_x_min, small_y_min, small_x_max, small_y_max = REGION_BBOX_UM
large_x_centre = 0.5 * (small_x_min + small_x_max)
large_y_centre = 0.5 * (small_y_min + small_y_max)
large_width = LARGE_SUBSET_SCALE * (small_x_max - small_x_min)
large_height = LARGE_SUBSET_SCALE * (small_y_max - small_y_min)
LARGE_REGION_BBOX_UM = (
    large_x_centre - 0.5 * large_width,
    large_y_centre - 0.5 * large_height,
    large_x_centre + 0.5 * large_width,
    large_y_centre + 0.5 * large_height,
)

LARGE_INCLUSIVE_SHAPE_NAME = "MOSAIK_local_convex_inclusive"
LARGE_INCLUSIVE_TABLE_NAME = "table_MOSAIK_local_convex_inclusive"
LARGE_INCLUSIVE_OUTPUT_ZARR = (
    OUTPUT_DIR / "P7513_local_convex_inclusive_5x.spatialdata.zarr"
)
LARGE_INCLUSIVE_MASK_PATH = (
    OUTPUT_DIR / "P7513_local_convex_inclusive_5x_cellpose.npy"
)
LARGE_INCLUSIVE_TRANSFORM_PATH = (
    OUTPUT_DIR / "P7513_local_convex_inclusive_5x_transform.json"
)
LARGE_INCLUSIVE_CELL_SUMMARY_PATH = (
    OUTPUT_DIR / "P7513_local_convex_inclusive_5x_cells.csv"
)
# ---------------------------------------------------------------------

large_x_min, large_y_min, large_x_max, large_y_max = LARGE_REGION_BBOX_UM
assert large_x_min < large_x_max and large_y_min < large_y_max

large_x_transform, large_y_transform = _resolve_dataset_mask_affine(
    "MERSCOPE", zarr_path=SOURCE_ZARR
)
large_full_mask = np.load(CELLPOSE_MASK_PATH, mmap_mode="r")
large_context_window = bbox_to_pixel_window(
    LARGE_REGION_BBOX_UM,
    large_x_transform,
    large_y_transform,
    large_full_mask.shape,
    padding_um=MASK_CONTEXT_UM,
)
large_core_window = bbox_to_pixel_window(
    LARGE_REGION_BBOX_UM,
    large_x_transform,
    large_y_transform,
    large_full_mask.shape,
)
large_row_start, large_row_stop, large_col_start, large_col_stop = (
    large_context_window
)
(
    large_core_row_start,
    large_core_row_stop,
    large_core_col_start,
    large_core_col_stop,
) = large_core_window

large_candidate_labels = np.unique(
    large_full_mask[
        large_core_row_start:large_core_row_stop,
        large_core_col_start:large_core_col_stop,
    ]
)
large_candidate_labels = large_candidate_labels[large_candidate_labels > 0]
large_mask_crop = np.asarray(
    large_full_mask[
        large_row_start:large_row_stop, large_col_start:large_col_stop
    ]
).copy()
large_truncated_labels = np.intersect1d(
    large_candidate_labels, edge_labels(large_mask_crop)
)
large_interior_labels = np.setdiff1d(
    large_candidate_labels, large_truncated_labels
)

large_source_proseg_table = ad.read_zarr(
    SOURCE_ZARR / "tables" / "table_MOSAIK_proseg"
)
large_known_original_ids = set(
    large_source_proseg_table.obs["original_cell_id"].astype(str)
)
large_selected_labels = np.asarray(
    [
        label
        for label in large_interior_labels
        if str(int(label)) in large_known_original_ids
    ],
    dtype=np.uint32,
)
if len(large_selected_labels) == 0:
    raise RuntimeError("The large box contains no complete Cellpose/ProSeg cells.")

large_mask_crop[~np.isin(large_mask_crop, large_selected_labels)] = 0
np.save(LARGE_INCLUSIVE_MASK_PATH, large_mask_crop)
large_crop_x_transform, large_crop_y_transform = shifted_crop_affine(
    large_x_transform,
    large_y_transform,
    large_row_start,
    large_col_start,
)
LARGE_INCLUSIVE_TRANSFORM_PATH.write_text(
    json.dumps(
        {
            "x_transform": list(large_crop_x_transform),
            "y_transform": list(large_crop_y_transform),
            "source_pixel_window": list(large_context_window),
            "requested_bbox_um": list(LARGE_REGION_BBOX_UM),
            "subset_scale": float(LARGE_SUBSET_SCALE),
        },
        indent=2,
    )
)

# Crop the standard MERSCOPE projection to the same context grid. Keeping
# the original source-pixel coordinates is essential: the interactive
# viewer uses them with the dataset pixel-to-micron affine when it
# rasterizes vector shapes into label outlines.
LARGE_IMAGE_CHANNELS = ("DAPI", "PolyT")
large_source_images_sdata = sd.read_zarr(SOURCE_ZARR, selection=("images",))
if len(large_source_images_sdata.images) == 0:
    raise RuntimeError("The source SpatialData contains no image elements.")
large_source_projection = build_merscope_z_projection(
    large_source_images_sdata.images
)
large_source_projection = large_source_projection.transpose("c", "y", "x")
if "c" not in large_source_projection.coords:
    raise RuntimeError("The MERSCOPE projection has no channel labels.")
large_available_channels = [
    str(value) for value in large_source_projection.coords["c"].values
]
large_channel_lookup = {
    channel.casefold(): channel for channel in large_available_channels
}
large_missing_image_channels = [
    channel
    for channel in LARGE_IMAGE_CHANNELS
    if channel.casefold() not in large_channel_lookup
]
if large_missing_image_channels:
    raise KeyError(
        f"Missing image channels {large_missing_image_channels}; "
        f"available channels are {large_available_channels}"
    )
large_source_channel_names = [
    large_channel_lookup[channel.casefold()] for channel in LARGE_IMAGE_CHANNELS
]
large_source_projection = large_source_projection.sel(
    c=large_source_channel_names
).assign_coords(c=list(LARGE_IMAGE_CHANNELS))

# Older source stores may omit explicit pixel coordinates. Add full-image
# indices before cropping so the crop begins at its original global pixel.
if "x" not in large_source_projection.coords:
    large_source_projection = large_source_projection.assign_coords(
        x=np.arange(large_source_projection.sizes["x"], dtype=np.int64)
    )
if "y" not in large_source_projection.coords:
    large_source_projection = large_source_projection.assign_coords(
        y=np.arange(large_source_projection.sizes["y"], dtype=np.int64)
    )
if (
    large_row_stop > large_source_projection.sizes["y"]
    or large_col_stop > large_source_projection.sizes["x"]
):
    raise RuntimeError(
        "The Cellpose crop window exceeds the MERSCOPE projection grid: "
        f"window={large_context_window}, image="
        f"{large_source_projection.sizes['y']}×"
        f"{large_source_projection.sizes['x']}"
    )
large_cropped_projection = large_source_projection.isel(
    y=slice(large_row_start, large_row_stop),
    x=slice(large_col_start, large_col_stop),
)
large_expected_image_shape = (
    large_row_stop - large_row_start,
    large_col_stop - large_col_start,
)
large_actual_image_shape = (
    int(large_cropped_projection.sizes["y"]),
    int(large_cropped_projection.sizes["x"]),
)
if large_actual_image_shape != large_expected_image_shape:
    raise RuntimeError(
        f"Unexpected cropped image shape {large_actual_image_shape}; "
        f"expected {large_expected_image_shape}"
    )
# Image2DModel.parse() normalises a crop to local pixel coordinates. Use
# the shifted crop affine so those local pixels retain their global micron
# location instead of being placed at the full image's zero origin.
large_image_transform = Affine(
    [
        list(large_crop_x_transform),
        list(large_crop_y_transform),
        [0.0, 0.0, 1.0],
    ],
    input_axes=("x", "y"),
    output_axes=("x", "y"),
)
large_cropped_image = Image2DModel.parse(
    large_cropped_projection,
    c_coords=list(LARGE_IMAGE_CHANNELS),
    rgb=None,
    chunks=(1, 1024, 1024),
    scale_factors=[2, 2, 2],
)
set_transformation(
    large_cropped_image,
    {"global": large_image_transform},
    set_all=True,
)

large_selected_original_ids = [
    str(int(label)) for label in large_selected_labels
]
large_proseg_row_mask = (
    large_source_proseg_table.obs["original_cell_id"]
    .astype(str)
    .isin(large_selected_original_ids)
)
large_selected_proseg_table = large_source_proseg_table[
    large_proseg_row_mask
].copy()
if DROP_MERSCOPE_CONTROLS:
    large_table_gene_names = large_selected_proseg_table.var_names.astype(str)
    large_table_gene_keep = ~pd.Series(large_table_gene_names).str.match(
        CONTROL_GENE_PATTERN, na=False
    ).to_numpy()
    large_selected_proseg_table = large_selected_proseg_table[
        :, large_table_gene_keep
    ].copy()
large_selected_proseg_ids = (
    large_selected_proseg_table.obs["cell"].astype(np.int64).tolist()
)

large_source_points = sd.read_zarr(
    SOURCE_ZARR, selection=("points",)
).points["transcripts"]
large_required_columns = {
    "observed_x", "observed_y", "gene", "assignment", "background"
}
large_missing_columns = large_required_columns.difference(large_source_points.columns)
if large_missing_columns:
    raise KeyError(
        f"Source transcript layer is missing {sorted(large_missing_columns)}"
    )

large_in_context = (
    large_source_points["observed_x"].between(
        large_x_min - MASK_CONTEXT_UM, large_x_max + MASK_CONTEXT_UM
    )
    & large_source_points["observed_y"].between(
        large_y_min - MASK_CONTEXT_UM, large_y_max + MASK_CONTEXT_UM
    )
)
large_assigned_to_selected = (
    large_source_points["assignment"]
    .isin(large_selected_proseg_ids)
    .fillna(False)
)
large_point_selector = large_in_context | large_assigned_to_selected
if DROP_MERSCOPE_CONTROLS:
    large_controls = large_source_points["gene"].astype("string").str.match(
        CONTROL_GENE_PATTERN, na=False
    )
    large_point_selector = large_point_selector & ~large_controls

large_algorithm_points_pdf = (
    large_source_points.loc[large_point_selector]
    .compute()
    .reset_index(drop=True)
)
if len(large_algorithm_points_pdf) == 0:
    raise RuntimeError("No transcripts were found for the large subsection.")

large_cellpose_polygons = _cellpose_polygons_in_microns(
    LARGE_INCLUSIVE_MASK_PATH,
    large_crop_x_transform,
    large_crop_y_transform,
)
large_cell_ids = sorted(large_cellpose_polygons, key=int)
large_cellpose_gdf = gpd.GeoDataFrame(
    {
        "cell_id": large_cell_ids,
        "cellpose_label": [int(value) for value in large_cell_ids],
        "geometry": [large_cellpose_polygons[value] for value in large_cell_ids],
    },
    geometry="geometry",
    index=pd.Index(large_cell_ids, name="cell_id_index"),
)
large_proseg_gdf = gpd.read_parquet(
    SOURCE_ZARR / "shapes" / "MOSAIK_proseg" / "shapes.parquet",
    filters=[("cell", "in", large_selected_proseg_ids)],
)
large_proseg_gdf = large_proseg_gdf.set_index(
    pd.Index(large_proseg_gdf["cell"].astype(np.int64), name="cell"),
    drop=False,
)

print(f"Large region: {LARGE_REGION_BBOX_UM} ({large_width:g} × {large_height:g} µm)")
print(f"Context mask crop: {large_mask_crop.shape} pixels")
print(
    f"Cropped image: channels={list(LARGE_IMAGE_CHANNELS)}, "
    f"shape={large_actual_image_shape}"
)
print(f"Selected complete cells: {len(large_cell_ids):,}")
print(f"Algorithm transcripts retained: {len(large_algorithm_points_pdf):,}")
print(f"Output: {LARGE_INCLUSIVE_OUTPUT_ZARR}")


In [ ]:
# Chosen inclusive condition from the 24-condition sweep.
LARGE_INCLUSIVE_MINIMUM_EXTERNAL_GROUP = 3
LARGE_INCLUSIVE_CHAIN_RADIUS_SCALE = 2.0
LARGE_INCLUSIVE_NEAR_SURFACE_RADIUS_FRACTION = 0.25
LARGE_INCLUSIVE_MAXIMUM_EXPANSION_RADIUS_FRACTION = 1.0
LARGE_INCLUSIVE_ATTACHMENT_ARC_WIDTH_SCALE = 0.5
LARGE_INCLUSIVE_ROUNDING_RADIUS_FRACTION = 0.15

large_assignment_map = {
    int(cell): str(original)
    for cell, original in zip(
        large_selected_proseg_table.obs["cell"],
        large_selected_proseg_table.obs["original_cell_id"],
        strict=True,
    )
}
large_x_column = (
    "observed_x" if "observed_x" in large_algorithm_points_pdf else "x"
)
large_y_column = (
    "observed_y" if "observed_y" in large_algorithm_points_pdf else "y"
)
large_proseg_cells = (
    pd.to_numeric(large_algorithm_points_pdf["assignment"], errors="coerce")
    .map(large_assignment_map)
    .astype("string")
)
large_foreground_valid = (
    ~large_algorithm_points_pdf["background"].astype(bool)
    & large_proseg_cells.notna()
)
large_foreground_source = large_algorithm_points_pdf.loc[
    large_foreground_valid, [large_x_column, large_y_column]
].copy()
large_foreground_source["_cell_id"] = (
    large_proseg_cells.loc[large_foreground_valid].to_numpy()
)
large_foreground_coordinates = {
    str(cell_id): group[[large_x_column, large_y_column]].to_numpy(np.float64)
    for cell_id, group in large_foreground_source.groupby("_cell_id", sort=False)
}

large_config_values = HYBRID_CONFIG.model_dump()
large_config_values.update(
    outlier_neighbors=2,
    outlier_mad_multiplier=2.0,
)
large_inclusive_config = ProsegHybridConfig.model_validate(large_config_values)
large_cell_results = {
    cell_id: build_local_convex_expansion_geometry(
        large_foreground_coordinates.get(
            cell_id, np.empty((0, 2), dtype=np.float64)
        ),
        large_cellpose_polygons[cell_id],
        large_inclusive_config,
        minimum_external_group=LARGE_INCLUSIVE_MINIMUM_EXTERNAL_GROUP,
        chain_radius_scale=LARGE_INCLUSIVE_CHAIN_RADIUS_SCALE,
        near_surface_radius_fraction=(
            LARGE_INCLUSIVE_NEAR_SURFACE_RADIUS_FRACTION
        ),
        maximum_expansion_radius_fraction=(
            LARGE_INCLUSIVE_MAXIMUM_EXPANSION_RADIUS_FRACTION
        ),
        attachment_arc_width_scale=(
            LARGE_INCLUSIVE_ATTACHMENT_ARC_WIDTH_SCALE
        ),
        rounding_radius_fraction=LARGE_INCLUSIVE_ROUNDING_RADIUS_FRACTION,
    )
    for cell_id in large_cell_ids
}
if any(result["unanchored_external"] for result in large_cell_results.values()):
    raise AssertionError("Strict transcript containment failed in the large run")

large_inclusive_polygons = [
    large_cell_results[cell_id]["geometry"] for cell_id in large_cell_ids
]
large_augmented_points_pdf = assign_transcripts_to_hybrid_masks(
    large_algorithm_points_pdf,
    large_inclusive_polygons,
    large_cell_ids,
    large_assignment_map,
    x_col=large_x_column,
    y_col=large_y_column,
    assignment_col="assignment",
)
large_augmented_points_pdf["proseg_cell_id"] = large_proseg_cells.to_numpy()

# Crop the exported point layer after geometry/assignment, not before.
large_export_selector = (
    large_augmented_points_pdf[large_x_column].between(
        large_x_min - MASK_CONTEXT_UM, large_x_max + MASK_CONTEXT_UM
    )
    & large_augmented_points_pdf[large_y_column].between(
        large_y_min - MASK_CONTEXT_UM, large_y_max + MASK_CONTEXT_UM
    )
)
large_export_points_pdf = (
    large_augmented_points_pdf.loc[large_export_selector]
    .copy()
    .reset_index(drop=True)
)

large_shape_records: list[dict[str, object]] = []
for cell_id in large_cell_ids:
    cell_result = large_cell_results[cell_id]
    cellpose_polygon = large_cellpose_polygons[cell_id]
    geometry = cell_result["geometry"]
    large_shape_records.append(
        {
            "cell_id": cell_id,
            "cellpose_label": int(cell_id),
            "proseg_foreground_transcripts": int(
                len(large_foreground_coordinates.get(cell_id, ()))
            ),
            "neighbor_scale_um": float(cell_result["neighbor_scale"]),
            "component_outliers": int(cell_result["component_outliers"]),
            "candidate_external": int(cell_result["candidate_external"]),
            "cap_rejected_external": int(
                cell_result["cap_rejected_external"]
            ),
            "unsupported_external": int(cell_result["unsupported_external"]),
            "near_surface_accepted": int(
                cell_result["near_surface_accepted"]
            ),
            "chain_accepted": int(cell_result["chain_accepted"]),
            "supported_external": int(cell_result["supported_external"]),
            "accepted_groups": int(cell_result["accepted_groups"]),
            "fallback_reason": str(cell_result["fallback_reason"]),
            "cellpose_area_um2": float(cellpose_polygon.area),
            "inclusive_area_um2": float(geometry.area),
            "area_ratio": float(geometry.area / cellpose_polygon.area),
            "geometry": geometry,
        }
    )
large_inclusive_gdf = gpd.GeoDataFrame(
    large_shape_records, geometry="geometry"
)
large_inclusive_gdf.index = pd.Index(
    large_inclusive_gdf["cell_id"].astype(str), name="cell_id"
)

large_originally_foreground = (
    ~large_augmented_points_pdf["background"].astype(bool)
    & large_augmented_points_pdf["proseg_cell_id"].notna()
)
large_summary = pd.Series(
    {
        "cells": len(large_cell_ids),
        "algorithm_transcripts": len(large_algorithm_points_pdf),
        "exported_transcripts": len(large_export_points_pdf),
        "supported_external": sum(
            result["supported_external"] for result in large_cell_results.values()
        ),
        "cap_rejected_external": sum(
            result["cap_rejected_external"]
            for result in large_cell_results.values()
        ),
        "hybrid_assigned_transcripts": int(
            large_augmented_points_pdf[HYBRID_ASSIGNMENT_COLUMN].notna().sum()
        ),
        "rejected_proseg_foreground": int(
            (
                large_originally_foreground
                & large_augmented_points_pdf[HYBRID_ASSIGNMENT_COLUMN].isna()
            ).sum()
        ),
        "rescued_proseg_background": int(
            (
                large_augmented_points_pdf["background"].astype(bool)
                & large_augmented_points_pdf[HYBRID_ASSIGNMENT_COLUMN].notna()
            ).sum()
        ),
        "ambiguous_overlap": int(
            (
                large_augmented_points_pdf[HYBRID_ASSIGNMENT_SOURCE_COLUMN]
                == "ambiguous_overlap"
            ).sum()
        ),
        "median_area_ratio": float(large_inclusive_gdf["area_ratio"].median()),
    },
    name="value",
)
display(large_summary.to_frame())


In [ ]:
from scipy import sparse

# Build a cell-by-gene count table from the final overlap-aware assignments.
large_genes = (
    large_selected_proseg_table.var["gene"].astype(str).tolist()
    if "gene" in large_selected_proseg_table.var.columns
    else large_selected_proseg_table.var_names.astype(str).tolist()
)
large_cell_to_index = {
    cell_id: index for index, cell_id in enumerate(large_cell_ids)
}
large_gene_to_index = {gene: index for index, gene in enumerate(large_genes)}
large_final_assigned = large_export_points_pdf[HYBRID_ASSIGNMENT_COLUMN].notna()
large_assigned_cells = (
    large_export_points_pdf.loc[large_final_assigned, HYBRID_ASSIGNMENT_COLUMN]
    .astype(str)
)
large_assigned_genes = (
    large_export_points_pdf.loc[large_final_assigned, "gene"].astype(str)
)
large_count_cell_indices = (
    large_assigned_cells.map(large_cell_to_index).fillna(-1).astype(int)
)
large_count_gene_indices = (
    large_assigned_genes.map(large_gene_to_index).fillna(-1).astype(int)
)
large_count_keep = (large_count_cell_indices >= 0) & (large_count_gene_indices >= 0)
large_counts = sparse.coo_matrix(
    (
        np.ones(int(large_count_keep.sum()), dtype=np.int64),
        (
            large_count_cell_indices.loc[large_count_keep].to_numpy(),
            large_count_gene_indices.loc[large_count_keep].to_numpy(),
        ),
    ),
    shape=(len(large_cell_ids), len(large_genes)),
).tocsr()

large_obs = large_inclusive_gdf.drop(columns="geometry").copy()
large_obs = large_obs.reindex(pd.Index(large_cell_ids, name="cell_id"))
large_obs["cell_id"] = large_obs.index.astype(str)
large_final_counts = large_assigned_cells.value_counts()
large_rescued_counts = (
    large_export_points_pdf.loc[
        large_export_points_pdf["background"].astype(bool)
        & large_final_assigned,
        HYBRID_ASSIGNMENT_COLUMN,
    ]
    .astype(str)
    .value_counts()
)
large_obs["hybrid_assigned_transcripts"] = (
    large_obs.index.to_series().map(large_final_counts).fillna(0).astype(np.int64)
)
large_obs["rescued_proseg_background"] = (
    large_obs.index.to_series().map(large_rescued_counts).fillna(0).astype(np.int64)
)
large_obs["region"] = pd.Categorical(
    [LARGE_INCLUSIVE_SHAPE_NAME] * len(large_obs),
    categories=[LARGE_INCLUSIVE_SHAPE_NAME],
)
large_var = pd.DataFrame(index=pd.Index(large_genes, dtype=str, name="gene"))
large_var["gene"] = large_var.index.astype(str)
large_inclusive_adata = ad.AnnData(X=large_counts, obs=large_obs, var=large_var)
large_inclusive_adata.uns["local_convex_inclusive"] = {
    "region_bbox_um": list(LARGE_REGION_BBOX_UM),
    "subset_scale": float(LARGE_SUBSET_SCALE),
    "minimum_external_group": LARGE_INCLUSIVE_MINIMUM_EXTERNAL_GROUP,
    "outlier_neighbors": 2,
    "outlier_mad_multiplier": 2.0,
    "chain_radius_scale": LARGE_INCLUSIVE_CHAIN_RADIUS_SCALE,
    "near_surface_radius_fraction": (
        LARGE_INCLUSIVE_NEAR_SURFACE_RADIUS_FRACTION
    ),
    "maximum_expansion_radius_fraction": (
        LARGE_INCLUSIVE_MAXIMUM_EXPANSION_RADIUS_FRACTION
    ),
    "attachment_arc_width_scale": LARGE_INCLUSIVE_ATTACHMENT_ARC_WIDTH_SCALE,
    "rounding_radius_fraction": LARGE_INCLUSIVE_ROUNDING_RADIUS_FRACTION,
}
large_inclusive_table = TableModel.parse(
    large_inclusive_adata,
    region=LARGE_INCLUSIVE_SHAPE_NAME,
    region_key="region",
    instance_key="cell_id",
)

large_selected_proseg_table.obs["region"] = pd.Categorical(
    ["MOSAIK_proseg_source"] * large_selected_proseg_table.n_obs,
    categories=["MOSAIK_proseg_source"],
)
large_source_table = TableModel.parse(
    large_selected_proseg_table,
    region="MOSAIK_proseg_source",
    region_key="region",
    instance_key="cell",
    overwrite_metadata=True,
)

large_npartitions = max(
    1, min(16, int(np.ceil(len(large_export_points_pdf) / 50_000)))
)
large_coordinate_mapping = {"x": large_x_column, "y": large_y_column}
large_z_column = (
    "observed_z"
    if "observed_z" in large_export_points_pdf
    else ("z" if "z" in large_export_points_pdf else None)
)
if large_z_column is not None:
    large_coordinate_mapping["z"] = large_z_column
large_parsed_points = PointsModel.parse(
    dd.from_pandas(large_export_points_pdf, npartitions=large_npartitions),
    coordinates=large_coordinate_mapping,
    feature_key="gene",
)
large_viewer_sdata = sd.SpatialData(
    images={MERSCOPE_ZPROJ_IMAGE_NAME: large_cropped_image},
    points={"transcripts": large_parsed_points},
    shapes={
        "MOSAIK_cellpose_source": ShapesModel.parse(large_cellpose_gdf),
        "MOSAIK_proseg_source": ShapesModel.parse(large_proseg_gdf),
        LARGE_INCLUSIVE_SHAPE_NAME: ShapesModel.parse(large_inclusive_gdf),
    },
    tables={
        "table": large_source_table,
        LARGE_INCLUSIVE_TABLE_NAME: large_inclusive_table,
    },
)

large_obs.drop(columns="region").to_csv(LARGE_INCLUSIVE_CELL_SUMMARY_PATH)
if LARGE_INCLUSIVE_OUTPUT_ZARR.exists():
    shutil.rmtree(LARGE_INCLUSIVE_OUTPUT_ZARR)
write_spatialdata_zarr(large_viewer_sdata, LARGE_INCLUSIVE_OUTPUT_ZARR)

# napari-spatialomics-viewer intentionally resolves its MERSCOPE image/grid
# affine from this conventional sidecar rather than trusting the transform
# stored on an image element. Save the inverse crop affine (microns -> local
# crop pixels) so images, transcripts, polygons, and generated labels share
# the same global coordinates in that viewer.
large_crop_pixel_to_micron = np.asarray(
    [
        list(large_crop_x_transform),
        list(large_crop_y_transform),
        [0.0, 0.0, 1.0],
    ],
    dtype=np.float64,
)
large_micron_to_crop_pixel = np.linalg.inv(large_crop_pixel_to_micron)
LARGE_INCLUSIVE_VIEWER_TRANSFORM_PATH = (
    LARGE_INCLUSIVE_OUTPUT_ZARR / "micron_to_mosaic_pixel_transform.csv"
)
np.savetxt(
    LARGE_INCLUSIVE_VIEWER_TRANSFORM_PATH,
    large_micron_to_crop_pixel,
    delimiter=" "
)
print(f"Wrote viewer SpatialData: {LARGE_INCLUSIVE_OUTPUT_ZARR}")
print(f"Wrote viewer crop transform: {LARGE_INCLUSIVE_VIEWER_TRANSFORM_PATH}")
print(f"Wrote cell diagnostics: {LARGE_INCLUSIVE_CELL_SUMMARY_PATH}")


In [ ]:
saved_large_inclusive = sd.read_zarr(LARGE_INCLUSIVE_OUTPUT_ZARR)
print(saved_large_inclusive)
print(f"Images: {list(saved_large_inclusive.images)}")
print(f"Shapes: {list(saved_large_inclusive.shapes)}")
print(f"Points: {list(saved_large_inclusive.points)}")
print(f"Tables: {list(saved_large_inclusive.tables)}")
saved_viewer_transform_path = (
    LARGE_INCLUSIVE_OUTPUT_ZARR / "micron_to_mosaic_pixel_transform.csv"
)
if not saved_viewer_transform_path.exists():
    raise FileNotFoundError(
        f"Viewer crop transform is missing: {saved_viewer_transform_path}"
    )
saved_micron_to_crop_pixel = np.loadtxt(saved_viewer_transform_path)
saved_crop_pixel_to_micron = np.linalg.inv(saved_micron_to_crop_pixel)
np.testing.assert_allclose(
    saved_crop_pixel_to_micron, large_crop_pixel_to_micron, rtol=0.0, atol=1e-9
)
print("Viewer crop pixel-to-micron transform:")
print(saved_crop_pixel_to_micron)
print(
    "Saved transcript rows:",
    f"{saved_large_inclusive.points['transcripts'].map_partitions(len).sum().compute():,}",
)


## Output contents

Load `OUTPUT_ZARR` directly in the interactive viewer. The principal elements are:

- `MOSAIK_cellpose_source`: selected raw Cellpose masks
- `MOSAIK_proseg_source`: original ProSeg masks for the same cells
- `MOSAIK_proseg_hybrid`: new transcript-supported masks
- `transcripts`: original ProSeg fields plus `hybrid_assignment`, `hybrid_background`, `hybrid_candidate_count`, and `hybrid_assignment_source`
- `table_MOSAIK_proseg_hybrid`: per-cell counts and geometry/outlier diagnostics

The store also retains the source `table` needed to reproduce the ProSeg-to-Cellpose ID mapping. All sweeps are intentionally in-memory: the original sweep writes `SWEEP_PLOT_PATH` plus `SWEEP_SUMMARY_PATH`, the fixed-support envelope writes `SUPPORTED_SWEEP_PLOT_PATH` plus `SUPPORTED_SWEEP_SUMMARY_PATH`, the distance-weighted method writes `DISTANCE_SWEEP_PLOT_PATH` plus `DISTANCE_SWEEP_SUMMARY_PATH`, and the focused tight-boundary cell writes `TIGHT_SWEEP_PLOT_PATH` plus `TIGHT_SWEEP_SUMMARY_PATH`. The local-convex experiment writes `CONVEX_SWEEP_SUMMARY_PATH` and three paginated figures listed in `convex_plot_paths`. The larger selected inclusive condition writes `LARGE_INCLUSIVE_OUTPUT_ZARR`, with a cropped two-channel `MERSCOPE_z_projection` image (DAPI and PolyT), Cellpose, ProSeg, inclusive-convex shapes, overlap-aware transcript assignments, and a cell-by-gene count table. The cropped image carries its shifted global pixel-to-micron transform, and the store includes `micron_to_mosaic_pixel_transform.csv` so napari-spatialomics-viewer applies the same crop origin when displaying images and rasterizing polygon labels. None overwrites the production hybrid branch or silently redefines its masks.

## Growth-only smoothing experiment

This section operates only on the already-created large inclusive masks in memory. It does not modify or save the SpatialData object. Each mask first has every internal hole filled, then undergoes round morphological closing to remove boundary features below a fixed physical scale. A small outward-only buffer rounds convex corners that closing cannot alter under strict containment. Newly added geometry is clipped to the existing Cellpose-relative expansion cap, while the complete pre-smoothing mask is retained as an exact lower bound.

The preview cell processes only cells whose allowed expansion cap can reach the requested display box. This gives the same masks and overlap-aware assignments inside the crop without recalculating the entire 5× subset on every parameter edit.

In [ ]:
def fill_all_polygon_holes(geometry: object) -> object:
    """Return the polygonal geometry with every interior ring filled."""
    parts = polygonal_parts(geometry)
    if not parts:
        return Polygon()
    filled = unary_union(
        [Polygon(part.exterior) for part in parts if not part.is_empty]
    )
    if not filled.is_valid:
        filled = filled.buffer(0)
    return filled


def count_polygon_holes(geometry: object) -> int:
    """Count interior rings across every polygonal part."""
    return int(sum(len(part.interiors) for part in polygonal_parts(geometry)))


def smooth_growth_only_geometry(
    current_geometry: object,
    cellpose_polygon: object,
    *,
    smoothing_radius_um: float,
    outward_rounding_um: float,
    maximum_expansion_radius_fraction: float,
    quad_segs: int = 32,
    containment_tolerance_um: float = 1.0e-5,
) -> dict[str, object]:
    """Smooth a mask by growth only while respecting its Cellpose cap."""
    smoothing_radius = float(smoothing_radius_um)
    outward_rounding = float(outward_rounding_um)
    maximum_expansion_fraction = float(maximum_expansion_radius_fraction)
    tolerance = float(containment_tolerance_um)
    segments = int(quad_segs)
    if smoothing_radius < 0.0:
        raise ValueError("smoothing_radius_um must be nonnegative")
    if outward_rounding < 0.0:
        raise ValueError("outward_rounding_um must be nonnegative")
    if maximum_expansion_fraction <= 0.0:
        raise ValueError("maximum_expansion_radius_fraction must be positive")
    if segments < 8:
        raise ValueError("quad_segs must be at least 8")
    if tolerance <= 0.0:
        raise ValueError("containment_tolerance_um must be positive")
    if current_geometry.is_empty or cellpose_polygon.is_empty:
        raise ValueError("Cannot smooth an empty current or Cellpose geometry")

    original = current_geometry
    holes_before = count_polygon_holes(original)
    base = fill_all_polygon_holes(original)
    equivalent_radius = float(np.sqrt(cellpose_polygon.area / np.pi))
    if not np.isfinite(equivalent_radius) or equivalent_radius <= 0.0:
        raise ValueError("Cellpose geometry must have positive finite area")

    cap_distance = maximum_expansion_fraction * equivalent_radius
    # Correct Shapely's inscribed buffer chords so the polygonal cap does not
    # accidentally cut the corresponding analytical Cellpose dilation.
    cap_buffer_distance = (cap_distance + tolerance) / np.cos(
        np.pi / (4.0 * segments)
    )
    cap_region = fill_all_polygon_holes(
        cellpose_polygon.buffer(cap_buffer_distance, quad_segs=segments)
    )

    if smoothing_radius > 0.0:
        closed = base.buffer(
            smoothing_radius, quad_segs=segments
        ).buffer(-smoothing_radius, quad_segs=segments)
        if closed.is_empty:
            closed = base
    else:
        closed = base
    closed = fill_all_polygon_holes(closed)
    rounded = (
        closed.buffer(outward_rounding, quad_segs=segments)
        if outward_rounding > 0.0 else closed
    )
    capped_candidate = rounded.intersection(cap_region)
    final_geometry = fill_all_polygon_holes(
        unary_union([base, capped_candidate])
    )
    if not final_geometry.is_valid:
        final_geometry = final_geometry.buffer(0)
    # Retain the original after topology repair as a final numerical safeguard.
    final_geometry = fill_all_polygon_holes(
        unary_union([original, final_geometry])
    )

    area_tolerance = max(tolerance * tolerance, original.area * 1.0e-10)
    missing_original_area = float(original.difference(final_geometry).area)
    if missing_original_area > area_tolerance:
        raise AssertionError(
            "Growth-only smoothing failed to contain the original mask: "
            f"missing area={missing_original_area:.6g} µm²"
        )
    newly_added = final_geometry.difference(base)
    cap_violation_area = float(newly_added.difference(cap_region).area)
    if cap_violation_area > area_tolerance:
        raise AssertionError(
            "Smoothing added geometry beyond the Cellpose expansion cap: "
            f"area={cap_violation_area:.6g} µm²"
        )
    holes_after = count_polygon_holes(final_geometry)
    if holes_after:
        raise AssertionError(f"Smoothed geometry still contains {holes_after} hole(s)")

    original_area = float(original.area)
    final_area = float(final_geometry.area)
    return {
        "geometry": final_geometry,
        "cap_region": cap_region,
        "smoothing_radius_um": smoothing_radius,
        "outward_rounding_um": outward_rounding,
        "equivalent_cellpose_radius_um": equivalent_radius,
        "original_area_um2": original_area,
        "smoothed_area_um2": final_area,
        "added_area_um2": float(final_geometry.difference(original).area),
        "area_growth_fraction": float(final_area / original_area - 1.0),
        "original_perimeter_um": float(original.length),
        "smoothed_perimeter_um": float(final_geometry.length),
        "perimeter_change_fraction": float(
            final_geometry.length / original.length - 1.0
        ),
        "holes_filled": holes_before,
        "missing_original_area_um2": missing_original_area,
        "cap_violation_area_um2": cap_violation_area,
    }


In [ ]:
# -------------------- Change parameters here -----------------------------
# Scientific smoothing arguments (all distances are fixed physical microns).
SMOOTHING_RADIUS_UM = 10.0
OUTWARD_ROUNDING_UM = 0.20
SMOOTHING_MAXIMUM_EXPANSION_RADIUS_FRACTION = (
    LARGE_INCLUSIVE_MAXIMUM_EXPANSION_RADIUS_FRACTION
)
SMOOTHING_QUAD_SEGS = 32
SMOOTHING_CONTAINMENT_TOLERANCE_UM = 1.0e-5

# Crop and display arguments.
SMOOTHING_PREVIEW_BBOX_UM = REGION_BBOX_UM
SMOOTHING_DAPI_CHANNEL = "DAPI"
SMOOTHING_DAPI_PERCENTILES = (1.0, 99.8)
SMOOTHING_DAPI_GAMMA = 0.70
SMOOTHING_DAPI_ALPHA = 0.00
SMOOTHING_MASK_ALPHA = 0.20
SMOOTHING_ADDED_AREA_ALPHA = 0.42
SMOOTHING_FIGSIZE = (17.0, 8.0)
SMOOTHING_INVERT_Y_AXIS = True
# -------------------------------------------------------------------------

preview_x_min, preview_y_min, preview_x_max, preview_y_max = (
    map(float, SMOOTHING_PREVIEW_BBOX_UM)
)
if not (preview_x_min < preview_x_max and preview_y_min < preview_y_max):
    raise ValueError("SMOOTHING_PREVIEW_BBOX_UM must be (xmin, ymin, xmax, ymax)")
preview_box = box(
    preview_x_min, preview_y_min, preview_x_max, preview_y_max
)

# Include every cell whose permitted expansion cap can reach the crop. This
# makes overlap-aware assignment inside the crop equivalent to an all-cell run.
smoothing_preview_cell_ids: list[str] = []
for cell_id in large_cell_ids:
    cellpose_polygon = large_cellpose_polygons[cell_id]
    equivalent_radius = float(np.sqrt(cellpose_polygon.area / np.pi))
    reachable_distance = (
        float(SMOOTHING_MAXIMUM_EXPANSION_RADIUS_FRACTION)
        * equivalent_radius
        + float(SMOOTHING_CONTAINMENT_TOLERANCE_UM)
    )
    if cellpose_polygon.distance(preview_box) <= reachable_distance:
        smoothing_preview_cell_ids.append(cell_id)
if not smoothing_preview_cell_ids:
    raise RuntimeError("No large-subset cells can reach the smoothing preview box")

smoothing_preview_results = {
    cell_id: smooth_growth_only_geometry(
        large_cell_results[cell_id]["geometry"],
        large_cellpose_polygons[cell_id],
        smoothing_radius_um=SMOOTHING_RADIUS_UM,
        outward_rounding_um=OUTWARD_ROUNDING_UM,
        maximum_expansion_radius_fraction=(
            SMOOTHING_MAXIMUM_EXPANSION_RADIUS_FRACTION
        ),
        quad_segs=SMOOTHING_QUAD_SEGS,
        containment_tolerance_um=SMOOTHING_CONTAINMENT_TOLERANCE_UM,
    )
    for cell_id in smoothing_preview_cell_ids
}
smoothing_preview_current_polygons = [
    large_cell_results[cell_id]["geometry"]
    for cell_id in smoothing_preview_cell_ids
]
smoothing_preview_smoothed_polygons = [
    smoothing_preview_results[cell_id]["geometry"]
    for cell_id in smoothing_preview_cell_ids
]
smoothing_preview_current_gdf = gpd.GeoDataFrame(
    {
        "cell_id": smoothing_preview_cell_ids,
        "geometry": smoothing_preview_current_polygons,
    },
    geometry="geometry",
)
smoothing_preview_smoothed_gdf = gpd.GeoDataFrame(
    {
        "cell_id": smoothing_preview_cell_ids,
        "geometry": smoothing_preview_smoothed_polygons,
    },
    geometry="geometry",
)
smoothing_preview_cellpose_gdf = gpd.GeoDataFrame(
    {
        "cell_id": smoothing_preview_cell_ids,
        "geometry": [
            large_cellpose_polygons[cell_id]
            for cell_id in smoothing_preview_cell_ids
        ],
    },
    geometry="geometry",
)
smoothing_preview_added_gdf = gpd.GeoDataFrame(
    {
        "cell_id": smoothing_preview_cell_ids,
        "geometry": [
            smoothing_preview_results[cell_id]["geometry"].difference(
                large_cell_results[cell_id]["geometry"]
            )
            for cell_id in smoothing_preview_cell_ids
        ],
    },
    geometry="geometry",
)

# Reassign only transcripts displayed in the crop, using exactly the same
# single-mask and ProSeg-informed overlap rules as the preceding large run.
smoothing_preview_point_selector = (
    large_algorithm_points_pdf[large_x_column].between(
        preview_x_min, preview_x_max
    )
    & large_algorithm_points_pdf[large_y_column].between(
        preview_y_min, preview_y_max
    )
)
smoothing_preview_source_points = (
    large_algorithm_points_pdf.loc[smoothing_preview_point_selector]
    .copy()
    .reset_index(drop=True)
)
smoothing_preview_assigned_points = assign_transcripts_to_hybrid_masks(
    smoothing_preview_source_points,
    smoothing_preview_smoothed_polygons,
    smoothing_preview_cell_ids,
    large_assignment_map,
    x_col=large_x_column,
    y_col=large_y_column,
    assignment_col="assignment",
)
smoothing_preview_assigned_points["proseg_cell_id"] = (
    pd.to_numeric(
        smoothing_preview_assigned_points["assignment"], errors="coerce"
    )
    .map(large_assignment_map)
    .astype("string")
)
smoothing_preview_current_points = large_augmented_points_pdf.loc[
    large_augmented_points_pdf[large_x_column].between(
        preview_x_min, preview_x_max
    )
    & large_augmented_points_pdf[large_y_column].between(
        preview_y_min, preview_y_max
    )
].copy()

# Stable cell colours shared by masks and both assignment panels.
smoothing_preview_color_ids = sorted(
    set(smoothing_preview_cell_ids)
    | set(
        smoothing_preview_current_points[HYBRID_ASSIGNMENT_COLUMN]
        .dropna().astype(str)
    )
    | set(
        smoothing_preview_assigned_points[HYBRID_ASSIGNMENT_COLUMN]
        .dropna().astype(str)
    ),
    key=int,
)
smoothing_preview_cmap = plt.colormaps["turbo"].resampled(
    max(1, len(smoothing_preview_color_ids))
)
smoothing_preview_palette_order = np.random.default_rng(7513).permutation(
    len(smoothing_preview_color_ids)
)
smoothing_preview_colors = {
    cell_id: smoothing_preview_cmap(int(palette_index))
    for cell_id, palette_index in zip(
        smoothing_preview_color_ids,
        smoothing_preview_palette_order,
        strict=True,
    )
}

# Lazily read only the requested DAPI crop from the already-prepared image.
preview_image_window = bbox_to_pixel_window(
    SMOOTHING_PREVIEW_BBOX_UM,
    large_crop_x_transform,
    large_crop_y_transform,
    large_actual_image_shape,
)
preview_row_start, preview_row_stop, preview_col_start, preview_col_stop = (
    preview_image_window
)
smoothing_preview_dapi = large_cropped_projection.sel(
    c=SMOOTHING_DAPI_CHANNEL
).isel(
    y=slice(preview_row_start, preview_row_stop),
    x=slice(preview_col_start, preview_col_stop),
)
smoothing_preview_dapi_values = np.asarray(
    smoothing_preview_dapi.compute(), dtype=np.float32
)
finite_dapi = smoothing_preview_dapi_values[
    np.isfinite(smoothing_preview_dapi_values)
]
if finite_dapi.size == 0:
    raise RuntimeError("The requested DAPI preview contains no finite pixels")
dapi_low, dapi_high = np.percentile(
    finite_dapi, SMOOTHING_DAPI_PERCENTILES
)
if dapi_high <= dapi_low:
    dapi_high = dapi_low + 1.0
smoothing_preview_dapi_display = np.clip(
    (smoothing_preview_dapi_values - dapi_low) / (dapi_high - dapi_low),
    0.0,
    1.0,
) ** float(SMOOTHING_DAPI_GAMMA)
if (
    abs(float(large_crop_x_transform[1])) > 1.0e-12
    or abs(float(large_crop_y_transform[0])) > 1.0e-12
):
    raise NotImplementedError("The preview imshow helper expects an axis-aligned affine")
preview_image_extent = (
    float(
        large_crop_x_transform[0] * preview_col_start
        + large_crop_x_transform[2]
    ),
    float(
        large_crop_x_transform[0] * preview_col_stop
        + large_crop_x_transform[2]
    ),
    float(
        large_crop_y_transform[1] * preview_row_start
        + large_crop_y_transform[2]
    ),
    float(
        large_crop_y_transform[1] * preview_row_stop
        + large_crop_y_transform[2]
    ),
)

smoothing_preview_diagnostics = pd.DataFrame(
    [
        {
            "cell_id": cell_id,
            **{
                key: value
                for key, value in smoothing_preview_results[cell_id].items()
                if key not in {"geometry", "cap_region"}
            },
        }
        for cell_id in smoothing_preview_cell_ids
    ]
).set_index("cell_id")

fig, smoothing_preview_axes = plt.subplots(
    1, 2, figsize=SMOOTHING_FIGSIZE,
    sharex=True, sharey=True, constrained_layout=True,
)
current_assigned_count = int(
    smoothing_preview_current_points[HYBRID_ASSIGNMENT_COLUMN].notna().sum()
)
smoothed_assigned_count = int(
    smoothing_preview_assigned_points[HYBRID_ASSIGNMENT_COLUMN].notna().sum()
)
for ax, mask_frame, panel_title in (
    (
        smoothing_preview_axes[0],
        smoothing_preview_current_gdf,
        f"Before smoothing | assigned={current_assigned_count:,}",
    ),
    (
        smoothing_preview_axes[1],
        smoothing_preview_smoothed_gdf,
        f"Growth-only smoothing | assigned={smoothed_assigned_count:,}",
    ),
):
    ax.imshow(
        smoothing_preview_dapi_display,
        cmap="gray",
        origin="lower",
        extent=preview_image_extent,
        alpha=float(SMOOTHING_DAPI_ALPHA),
        interpolation="nearest",
        zorder=0,
    )
    plot_polygons_by_cell(
        ax,
        mask_frame,
        mask_frame["cell_id"],
        smoothing_preview_colors,
        alpha=float(SMOOTHING_MASK_ALPHA),
    )
    smoothing_preview_cellpose_gdf.boundary.plot(
        ax=ax, color="white", linestyle="--", linewidth=0.8,
        alpha=0.8, zorder=4,
    )
    ax.set_title(panel_title)
    ax.set_xlim(preview_x_min, preview_x_max)
    ax.set_ylim(preview_y_min, preview_y_max)
    if SMOOTHING_INVERT_Y_AXIS:
        ax.set_ylim(preview_y_max, preview_y_min)
    ax.set_aspect("equal")
    ax.set_xlabel("observed x (µm)")
    ax.set_ylabel("observed y (µm)")

visible_added = smoothing_preview_added_gdf.loc[
    ~smoothing_preview_added_gdf.geometry.is_empty
]
if len(visible_added):
    added_colors = [
        smoothing_preview_colors[str(cell_id)]
        for cell_id in visible_added["cell_id"]
    ]
    visible_added.plot(
        ax=smoothing_preview_axes[1],
        color=added_colors,
        edgecolor="white",
        linewidth=0.35,
        alpha=float(SMOOTHING_ADDED_AREA_ALPHA),
        zorder=1,
    )
smoothing_preview_current_gdf.boundary.plot(
    ax=smoothing_preview_axes[1],
    color="white",
    linestyle=":",
    linewidth=0.65,
    alpha=0.75,
    zorder=5,
)
# Add transcripts last so the diagnostic mask fills never obscure them.
plot_sweep_transcripts(
    smoothing_preview_axes[0],
    smoothing_preview_current_points,
    smoothing_preview_colors,
)
plot_sweep_transcripts(
    smoothing_preview_axes[1],
    smoothing_preview_assigned_points,
    smoothing_preview_colors,
)
fig.suptitle(
    "Growth-only mask smoothing preview\n"
    f"closing radius={SMOOTHING_RADIUS_UM:g} µm; "
    f"outward rounding={OUTWARD_ROUNDING_UM:g} µm; "
    f"cap={SMOOTHING_MAXIMUM_EXPANSION_RADIUS_FRACTION:g}R"
)
plt.show()

smoothing_preview_summary = pd.Series(
    {
        "preview_cells": len(smoothing_preview_cell_ids),
        "preview_transcripts": len(smoothing_preview_assigned_points),
        "assigned_before": current_assigned_count,
        "assigned_after": smoothed_assigned_count,
        "median_area_growth_percent": float(
            100.0 * smoothing_preview_diagnostics["area_growth_fraction"].median()
        ),
        "maximum_area_growth_percent": float(
            100.0 * smoothing_preview_diagnostics["area_growth_fraction"].max()
        ),
        "median_perimeter_change_percent": float(
            100.0
            * smoothing_preview_diagnostics[
                "perimeter_change_fraction"
            ].median()
        ),
        "holes_filled": int(
            smoothing_preview_diagnostics["holes_filled"].sum()
        ),
        "rescued_original_background": int(
            (
                smoothing_preview_assigned_points["background"].astype(bool)
                & smoothing_preview_assigned_points[
                    HYBRID_ASSIGNMENT_COLUMN
                ].notna()
            ).sum()
        ),
        "ambiguous_overlap": int(
            (
                smoothing_preview_assigned_points[
                    HYBRID_ASSIGNMENT_SOURCE_COLUMN
                ]
                == "ambiguous_overlap"
            ).sum()
        ),
    },
    name="value",
)
display(smoothing_preview_summary.to_frame())
display(
    smoothing_preview_diagnostics[
        [
            "original_area_um2",
            "smoothed_area_um2",
            "area_growth_fraction",
            "perimeter_change_fraction",
            "holes_filled",
        ]
    ].describe(percentiles=[0.1, 0.5, 0.9])
)
